In [1]:
from pathlib import Path
import json
import re
import time
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM


# ============================================================
# SETTINGS
# ============================================================

MODEL_ID = "Qwen/Qwen3-8B"

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

consultation_id = "day1_consultation01"

transcript_path = (
    project
    / "results/asr/whisper_large_v3/datalab_pilot"
    / f"{consultation_id}_transcript.txt"
)

output_dir = (
    project
    / "results/nlp/qwen3_8b/compatibility_test"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 75)
print("QWEN3-8B CLINICAL EXTRACTION COMPATIBILITY TEST")
print("=" * 75)

print(
    "Transcript exists:",
    transcript_path.exists()
)

transcript = transcript_path.read_text(
    encoding="utf-8"
)

print(
    "Transcript characters:",
    len(transcript)
)


# ============================================================
# LOAD QWEN
# ============================================================

print("\nLoading Qwen3-8B...")

load_start = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto"
)

model.eval()

load_seconds = time.time() - load_start

print("Model loaded successfully")
print(
    "Load time:",
    round(load_seconds, 2),
    "seconds"
)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)


# ============================================================
# SAME COMPACT EXTRACTION SCHEMA
# ============================================================

prompt = f"""
Extract the clinically important facts from this medical consultation.

Use ONLY facts explicitly stated in the transcript.

Rules:
- Do not infer or invent information.
- Preserve important negations.
- Preserve numbers, durations, doses and frequencies.
- Combine duplicate or closely related information.
- Extract approximately 15 to 25 important clinical facts.
- Each evidence_quote must be a short quote from the transcript.
- status must be "present" or "absent".
- Return valid JSON only.
- No explanation, reasoning, Markdown or commentary.

Allowed categories:
presenting_complaint
symptom
temporal_detail
medication
allergy
medical_history
family_history
social_history
assessment
plan
safety_netting

Return exactly this structure:

{{
  "consultation_id": "{consultation_id}",
  "clinical_facts": [
    {{
      "category": "",
      "fact": "",
      "status": "present",
      "evidence_quote": ""
    }}
  ]
}}

TRANSCRIPT:

{transcript}
"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]


# ============================================================
# DISABLE THINKING
# ============================================================

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

input_tokens = (
    inputs["input_ids"].shape[-1]
)

print(
    "\nInput tokens:",
    input_tokens
)


# ============================================================
# GENERATE
# ============================================================

print(
    "\nGenerating structured clinical extraction..."
)

torch.cuda.reset_peak_memory_stats()

generation_start = time.time()

with torch.inference_mode():

    output_ids = model.generate(
        **inputs,
        max_new_tokens=2500,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generation_seconds = (
    time.time()
    - generation_start
)

generated_ids = output_ids[
    0,
    input_tokens:
]

generated_tokens = len(
    generated_ids
)

response = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

peak_gpu_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)


# ============================================================
# CLEAN RESPONSE
# ============================================================

cleaned = response.strip()

cleaned = re.sub(
    r"^```(?:json)?\s*",
    "",
    cleaned,
    flags=re.IGNORECASE
)

cleaned = re.sub(
    r"\s*```$",
    "",
    cleaned
).strip()

first = cleaned.find("{")
last = cleaned.rfind("}")

if first >= 0 and last > first:

    json_text = cleaned[
        first:last + 1
    ]

else:

    json_text = cleaned


# ============================================================
# JSON VALIDATION
# ============================================================

json_valid = False
parsed = None
json_error = None

try:

    parsed = json.loads(
        json_text
    )

    json_valid = True

except Exception as e:

    json_error = str(e)


# ============================================================
# EVIDENCE CHECK
# ============================================================

def norm(text):

    text = str(text).lower()

    text = re.sub(
        r"[^\w\s]",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


evidence_total = 0
evidence_matched = 0

unmatched = []

if json_valid:

    normalized_transcript = norm(
        transcript
    )

    facts = parsed.get(
        "clinical_facts",
        []
    )

    for i, item in enumerate(
        facts,
        start=1
    ):

        quote = item.get(
            "evidence_quote",
            ""
        )

        if quote:

            evidence_total += 1

            if (
                norm(quote)
                in normalized_transcript
            ):

                evidence_matched += 1

            else:

                unmatched.append({
                    "number": i,
                    "fact": item.get(
                        "fact"
                    ),
                    "quote": quote
                })


# ============================================================
# SAVE RESULTS
# ============================================================

raw_path = (
    output_dir
    / f"{consultation_id}_qwen3_raw.txt"
)

raw_path.write_text(
    response,
    encoding="utf-8"
)

json_path = (
    output_dir
    / f"{consultation_id}_qwen3_extraction.json"
)

if json_valid:

    with open(
        json_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            parsed,
            f,
            indent=2,
            ensure_ascii=False
        )


metadata = {
    "consultation_id":
        consultation_id,

    "model":
        MODEL_ID,

    "thinking_enabled":
        False,

    "input_tokens":
        input_tokens,

    "generated_tokens":
        generated_tokens,

    "generation_seconds":
        round(
            generation_seconds,
            2
        ),

    "peak_gpu_gb":
        round(
            peak_gpu_gb,
            2
        ),

    "json_valid":
        json_valid,

    "json_error":
        json_error,

    "facts_extracted":
        (
            len(
                parsed.get(
                    "clinical_facts",
                    []
                )
            )
            if json_valid
            else 0
        ),

    "evidence_total":
        evidence_total,

    "evidence_matched":
        evidence_matched,

    "evidence_grounding_percent":
        (
            round(
                evidence_matched
                / evidence_total
                * 100,
                2
            )
            if evidence_total
            else None
        )
}

metadata_path = (
    output_dir
    / f"{consultation_id}_qwen3_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n" + "=" * 75)
print("QWEN3-8B COMPATIBILITY TEST RESULT")
print("=" * 75)

print(
    "JSON valid:",
    json_valid
)

print(
    "Generated tokens:",
    generated_tokens
)

print(
    "Hit token limit:",
    generated_tokens >= 2500
)

print(
    "Generation time:",
    round(
        generation_seconds,
        2
    ),
    "seconds"
)

print(
    "Peak GPU memory:",
    round(
        peak_gpu_gb,
        2
    ),
    "GB"
)

if json_valid:

    print(
        "Clinical facts extracted:",
        len(
            parsed.get(
                "clinical_facts",
                []
            )
        )
    )

    print(
        "Evidence grounding:",
        f"{evidence_matched}/{evidence_total}",
        "=",
        (
            round(
                evidence_matched
                / evidence_total
                * 100,
                2
            )
            if evidence_total
            else 0
        ),
        "%"
    )

    print("\nSTRUCTURED JSON")
    print("-" * 75)

    print(
        json.dumps(
            parsed,
            indent=2,
            ensure_ascii=False
        )
    )

    if unmatched:

        print("\nUNGROUNDED ITEMS")

        for item in unmatched:
            print(item)

else:

    print(
        "JSON error:",
        json_error
    )

    print(
        "\nRaw response:"
    )

    print(response)

print("\nSaved:")
print(raw_path)

if json_valid:
    print(json_path)

print(metadata_path)


QWEN3-8B CLINICAL EXTRACTION COMPATIBILITY TEST
Transcript exists: True
Transcript characters: 6975

Loading Qwen3-8B...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00001-of-00005.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded successfully
Load time: 142.64 seconds
GPU: NVIDIA H200 NVL

Input tokens: 1920

Generating structured clinical extraction...

QWEN3-8B COMPATIBILITY TEST RESULT
JSON valid: True
Generated tokens: 1719
Hit token limit: False
Generation time: 39.46 seconds
Peak GPU memory: 15.8 GB
Clinical facts extracted: 28
Evidence grounding: 25/28 = 89.29 %

STRUCTURED JSON
---------------------------------------------------------------------------
{
  "consultation_id": "day1_consultation01",
  "clinical_facts": [
    {
      "category": "presenting_complaint",
      "fact": "Diarrhoea",
      "status": "present",
      "evidence_quote": "I've just had some diarrhoea for the last three days and it's been affecting me."
    },
    {
      "category": "symptom",
      "fact": "Loose and watery stool",
      "status": "present",
      "evidence_quote": "It's like loose and watery stool, going to the toilet quite often"
    },
    {
      "category": "temporal_detail",
      "fact": "Durat

In [2]:
import json
import re
import time
import torch
from pathlib import Path

print("=" * 75)
print("QWEN3-8B FINAL STRUCTURED EXTRACTION TEST")
print("=" * 75)

consultation_id = "day1_consultation01"

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

transcript_path = (
    project
    / "results/asr/whisper_large_v3/datalab_pilot"
    / f"{consultation_id}_transcript.txt"
)

output_dir = (
    project
    / "results/nlp/qwen3_8b/final_schema_test"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

transcript = transcript_path.read_text(
    encoding="utf-8"
)

# ============================================================
# FINAL STRICT PROMPT
# ============================================================

prompt = f"""
Extract the most clinically important facts from this
doctor-patient consultation.

Use ONLY information explicitly stated in the transcript.

STRICT REQUIREMENTS:

1. Return between 15 and 20 clinical facts TOTAL.
2. Do not duplicate the same clinical information.
3. Read the entire transcript, including the doctor's
   assessment, treatment plan and follow-up advice.
4. Prioritize clinically important information over ordinary
   social details.
5. Never invent or infer information.

CATEGORY DEFINITIONS:

presenting_complaint
- Main reason for consultation.

symptom
- Positive or explicitly denied symptom/finding.

temporal_detail
- Important onset, duration, frequency or severity.

medication
- Current medication only.

allergy
- Drug or substance allergy only.

medical_history
- Diagnosed past or current medical condition only.

exposure_history
- Relevant food, travel, infectious or environmental exposure.

social_history
- Smoking, alcohol, occupation or living situation only if
  clinically relevant.

assessment
- Doctor's working diagnosis or differential.

plan
- Treatment, medication advice, investigations or management.

safety_netting
- Follow-up instructions, return precautions or escalation advice.

STATUS RULE:

Use "present" when the clinical fact is affirmed.

Use "absent" when the patient explicitly denies the clinical fact.

Examples:

Fact: "Blood in vomit"
status: "absent"

Fact: "Smoking"
status: "absent"

Fact: "Asthma"
status: "present"

Do NOT write:
"No blood in vomit" with status "present".

EVIDENCE RULE:

Every evidence_quote MUST:
- be copied exactly from ONE continuous span of the transcript
- contain approximately 3 to 18 words
- not contain ellipses
- not combine separate parts of the transcript
- not paraphrase or correct the transcript

Do NOT use a doctor's question as evidence unless the patient's
answer confirming or denying the fact is included in the same
short continuous quote.

BALANCE THE OUTPUT:

Include:
- main complaint
- major symptoms
- clinically important negations
- important duration/frequency/numerical detail
- relevant history/medications/allergies/exposure
- assessment if stated
- important treatment plan
- follow-up or safety-netting

Do not spend most of the 15-20 facts only on symptoms.

Return JSON only.
No reasoning.
No explanation.
No Markdown.

Required structure:

{{
  "consultation_id": "{consultation_id}",
  "clinical_facts": [
    {{
      "category": "",
      "fact": "",
      "status": "present",
      "importance": "critical",
      "evidence_quote": ""
    }}
  ]
}}

importance must be exactly:
"critical" or "important"

TRANSCRIPT:

{transcript}
"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

input_tokens = inputs["input_ids"].shape[-1]

print("Input tokens:", input_tokens)

# ============================================================
# GENERATION
# ============================================================

torch.cuda.reset_peak_memory_stats()

start = time.time()

with torch.inference_mode():

    output_ids = model.generate(
        **inputs,
        max_new_tokens=2200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generation_seconds = time.time() - start

generated_ids = output_ids[
    0,
    input_tokens:
]

generated_tokens = len(generated_ids)

response = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

peak_gpu_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

# ============================================================
# CLEAN + PARSE
# ============================================================

cleaned = response.strip()

cleaned = re.sub(
    r"^```(?:json)?\s*",
    "",
    cleaned,
    flags=re.IGNORECASE
)

cleaned = re.sub(
    r"\s*```$",
    "",
    cleaned
).strip()

first = cleaned.find("{")
last = cleaned.rfind("}")

if first >= 0 and last > first:
    json_text = cleaned[first:last + 1]
else:
    json_text = cleaned

json_valid = False
parsed = None
json_error = None

try:
    parsed = json.loads(json_text)
    json_valid = True

except Exception as e:
    json_error = str(e)

# ============================================================
# VALIDATION
# ============================================================

allowed_categories = {
    "presenting_complaint",
    "symptom",
    "temporal_detail",
    "medication",
    "allergy",
    "medical_history",
    "exposure_history",
    "social_history",
    "assessment",
    "plan",
    "safety_netting"
}

def norm(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

facts = []
unmatched = []
invalid_status = []
invalid_category = []

evidence_total = 0
evidence_matched = 0

category_counts = {}

if json_valid:

    facts = parsed.get(
        "clinical_facts",
        []
    )

    normalized_transcript = norm(
        transcript
    )

    for i, item in enumerate(
        facts,
        start=1
    ):

        category = item.get(
            "category",
            ""
        )

        status = item.get(
            "status",
            ""
        )

        quote = item.get(
            "evidence_quote",
            ""
        )

        category_counts[category] = (
            category_counts.get(
                category,
                0
            ) + 1
        )

        if category not in allowed_categories:
            invalid_category.append(
                (i, category)
            )

        if status not in {
            "present",
            "absent"
        }:
            invalid_status.append(
                (i, status)
            )

        if quote:

            evidence_total += 1

            if norm(quote) in normalized_transcript:
                evidence_matched += 1

            else:
                unmatched.append({
                    "number": i,
                    "category": category,
                    "fact": item.get(
                        "fact"
                    ),
                    "status": status,
                    "quote": quote
                })

# ============================================================
# SAVE
# ============================================================

raw_path = (
    output_dir
    / f"{consultation_id}_qwen3_final_raw.txt"
)

raw_path.write_text(
    response,
    encoding="utf-8"
)

json_path = (
    output_dir
    / f"{consultation_id}_qwen3_final_extraction.json"
)

if json_valid:

    with open(
        json_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            parsed,
            f,
            indent=2,
            ensure_ascii=False
        )

metadata = {
    "consultation_id":
        consultation_id,

    "model":
        "Qwen/Qwen3-8B",

    "thinking_enabled":
        False,

    "json_valid":
        json_valid,

    "json_error":
        json_error,

    "facts_extracted":
        len(facts),

    "target_fact_range":
        "15-20",

    "input_tokens":
        input_tokens,

    "generated_tokens":
        generated_tokens,

    "generation_seconds":
        round(
            generation_seconds,
            2
        ),

    "peak_gpu_gb":
        round(
            peak_gpu_gb,
            2
        ),

    "evidence_total":
        evidence_total,

    "evidence_matched":
        evidence_matched,

    "evidence_grounding_percent":
        (
            round(
                evidence_matched
                / evidence_total
                * 100,
                2
            )
            if evidence_total
            else None
        ),

    "invalid_status_count":
        len(invalid_status),

    "invalid_category_count":
        len(invalid_category),

    "category_counts":
        category_counts
}

metadata_path = (
    output_dir
    / f"{consultation_id}_qwen3_final_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )

# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 75)
print("QWEN3-8B FINAL SCHEMA RESULT")
print("=" * 75)

print(
    "JSON valid:",
    json_valid
)

print(
    "Generated tokens:",
    generated_tokens
)

print(
    "Hit token limit:",
    generated_tokens >= 2200
)

if json_valid:

    print(
        "Clinical facts:",
        len(facts)
    )

    print(
        "Target 15-20 satisfied:",
        15 <= len(facts) <= 20
    )

    print(
        "Evidence grounding:",
        f"{evidence_matched}/{evidence_total}",
        "=",
        round(
            evidence_matched
            / evidence_total
            * 100,
            2
        )
        if evidence_total
        else 0,
        "%"
    )

    print(
        "Invalid statuses:",
        len(invalid_status)
    )

    print(
        "Invalid categories:",
        len(invalid_category)
    )

    print("\nCATEGORY COUNTS")

    for category, count in sorted(
        category_counts.items()
    ):
        print(
            f"{category}: {count}"
        )

    print("\nSTRUCTURED JSON")
    print("-" * 75)

    print(
        json.dumps(
            parsed,
            indent=2,
            ensure_ascii=False
        )
    )

    if unmatched:

        print("\nUNGROUNDED ITEMS")

        for item in unmatched:
            print(item)

else:

    print(
        "JSON error:",
        json_error
    )

print(
    "\nGeneration time:",
    round(
        generation_seconds,
        2
    ),
    "seconds"
)

print(
    "Peak GPU:",
    round(
        peak_gpu_gb,
        2
    ),
    "GB"
)

print("\nSaved:")
print(raw_path)

if json_valid:
    print(json_path)

print(metadata_path)


QWEN3-8B FINAL STRUCTURED EXTRACTION TEST
Input tokens: 2297

QWEN3-8B FINAL SCHEMA RESULT
JSON valid: True
Generated tokens: 1094
Hit token limit: False
Clinical facts: 20
Target 15-20 satisfied: True
Evidence grounding: 19/20 = 95.0 %
Invalid statuses: 0
Invalid categories: 0

CATEGORY COUNTS
assessment: 1
exposure_history: 1
medical_history: 1
plan: 5
presenting_complaint: 1
social_history: 2
symptom: 9

STRUCTURED JSON
---------------------------------------------------------------------------
{
  "consultation_id": "day1_consultation01",
  "clinical_facts": [
    {
      "category": "presenting_complaint",
      "fact": "Diarrhoea",
      "status": "present",
      "importance": "critical",
      "evidence_quote": "I've just had some diarrhoea for the last three days and it's been affecting me."
    },
    {
      "category": "symptom",
      "fact": "Loose and watery stool",
      "status": "present",
      "importance": "critical",
      "evidence_quote": "it's like loose and wa

In [3]:
from pathlib import Path
import json
import re
from difflib import SequenceMatcher

print("=" * 75)
print("QWEN3-8B CLINICAL EXTRACTION POST-PROCESSING")
print("=" * 75)

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

consultation_id = "day1_consultation01"

input_json = (
    project
    / "results/nlp/qwen3_8b/final_schema_test"
    / f"{consultation_id}_qwen3_final_extraction.json"
)

transcript_path = (
    project
    / "results/asr/whisper_large_v3/datalab_pilot"
    / f"{consultation_id}_transcript.txt"
)

output_dir = (
    project
    / "results/nlp/qwen3_8b/canonical_extraction"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_json = (
    output_dir
    / f"{consultation_id}_qwen3_canonical.json"
)

with open(
    input_json,
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)

transcript = transcript_path.read_text(
    encoding="utf-8"
)


# ============================================================
# HELPERS
# ============================================================

def normalize_words(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def repair_quote(quote, transcript):

    # Already exact
    if quote in transcript:
        return quote, 1.0, False

    transcript_words = transcript.split()
    quote_words = quote.split()

    target = normalize_words(quote)

    n = len(quote_words)

    best_score = 0
    best_text = None

    min_size = max(3, n - 5)
    max_size = min(
        len(transcript_words),
        n + 6
    )

    for window_size in range(
        min_size,
        max_size + 1
    ):

        for start in range(
            0,
            len(transcript_words)
            - window_size + 1
        ):

            candidate = " ".join(
                transcript_words[
                    start:start + window_size
                ]
            )

            score = SequenceMatcher(
                None,
                target,
                normalize_words(candidate)
            ).ratio()

            if score > best_score:
                best_score = score
                best_text = candidate

    if best_score >= 0.78:
        return best_text, best_score, True

    return quote, best_score, False


def standardize_negative_fact(
    fact,
    status,
    evidence
):

    lower_fact = fact.lower().strip()
    lower_evidence = evidence.lower()

    negative = False

    if lower_fact.startswith("no "):
        negative = True

    if lower_fact in {
        "non-smoker",
        "non smoker",
        "non-alcoholic",
        "non alcoholic"
    }:
        negative = True

    explicit_negative_patterns = [
        "no blood",
        "don't smoke",
        "do not smoke",
        "don't drink alcohol",
        "do not drink alcohol",
        "denies ",
        "no vomiting",
        "no fever",
        "no pain",
        "no discharge"
    ]

    if any(
        phrase in lower_evidence
        for phrase in explicit_negative_patterns
    ):
        negative = True

    if negative:
        status = "absent"

        if lower_fact.startswith("no "):
            fact = fact[3:].strip()

            if fact:
                fact = (
                    fact[0].upper()
                    + fact[1:]
                )

        elif lower_fact in {
            "non-smoker",
            "non smoker"
        }:
            fact = "Smoking"

        elif lower_fact in {
            "non-alcoholic",
            "non alcoholic"
        }:
            fact = "Alcohol consumption"

    return fact, status


# ============================================================
# POST-PROCESS
# ============================================================

facts = data.get(
    "clinical_facts",
    []
)

status_corrections = 0
quote_repairs = 0

repair_log = []

for number, item in enumerate(
    facts,
    start=1
):

    original_fact = item.get(
        "fact",
        ""
    )

    original_status = item.get(
        "status",
        ""
    )

    original_quote = item.get(
        "evidence_quote",
        ""
    )

    new_fact, new_status = (
        standardize_negative_fact(
            original_fact,
            original_status,
            original_quote
        )
    )

    if (
        new_fact != original_fact
        or new_status != original_status
    ):
        status_corrections += 1

    item["fact"] = new_fact
    item["status"] = new_status

    repaired_quote, score, repaired = (
        repair_quote(
            original_quote,
            transcript
        )
    )

    if repaired:
        quote_repairs += 1

        item["evidence_quote"] = (
            repaired_quote
        )

        repair_log.append({
            "fact_number": number,
            "fact": new_fact,
            "similarity": round(
                score,
                3
            ),
            "original_quote":
                original_quote,
            "repaired_quote":
                repaired_quote
        })


# ============================================================
# FINAL STRICT VALIDATION
# ============================================================

valid_categories = {
    "presenting_complaint",
    "symptom",
    "temporal_detail",
    "medication",
    "allergy",
    "medical_history",
    "exposure_history",
    "social_history",
    "assessment",
    "plan",
    "safety_netting"
}

valid_status = {
    "present",
    "absent"
}

valid_importance = {
    "critical",
    "important"
}

evidence_exact = 0

validation_errors = []

for number, item in enumerate(
    facts,
    start=1
):

    category = item.get(
        "category",
        ""
    )

    status = item.get(
        "status",
        ""
    )

    importance = item.get(
        "importance",
        ""
    )

    quote = item.get(
        "evidence_quote",
        ""
    )

    if category not in valid_categories:
        validation_errors.append(
            f"Fact {number}: invalid category"
        )

    if status not in valid_status:
        validation_errors.append(
            f"Fact {number}: invalid status"
        )

    if importance not in valid_importance:
        validation_errors.append(
            f"Fact {number}: invalid importance"
        )

    if quote in transcript:
        evidence_exact += 1
    else:
        validation_errors.append(
            f"Fact {number}: evidence not exact"
        )


# ============================================================
# SAVE CANONICAL OUTPUT
# ============================================================

canonical = {
    "consultation_id":
        consultation_id,

    "model":
        "Qwen/Qwen3-8B",

    "post_processing":
        True,

    "clinical_facts":
        facts
}

with open(
    output_json,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        canonical,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# RESULTS
# ============================================================

print(
    "Clinical facts:",
    len(facts)
)

print(
    "Status/fact corrections:",
    status_corrections
)

print(
    "Evidence quote repairs:",
    quote_repairs
)

print(
    "Exact grounded evidence:",
    f"{evidence_exact}/{len(facts)}"
)

print(
    "Evidence grounding:",
    round(
        evidence_exact
        / len(facts)
        * 100,
        2
    ),
    "%"
)

print(
    "Validation errors:",
    len(validation_errors)
)

print("\nABSENT / NEGATED FACTS")
print("-" * 75)

for item in facts:

    if item["status"] == "absent":

        print(
            f'{item["fact"]} | '
            f'{item["evidence_quote"]}'
        )

if repair_log:

    print("\nEVIDENCE REPAIRS")
    print("-" * 75)

    for item in repair_log:

        print(
            json.dumps(
                item,
                indent=2,
                ensure_ascii=False
            )
        )

if validation_errors:

    print("\nVALIDATION ERRORS")
    print("-" * 75)

    for error in validation_errors:
        print(error)

print("\nSaved:")
print(output_json)

print("\n" + "=" * 75)

if len(validation_errors) == 0:
    print(
        "CANONICAL EXTRACTION VALIDATION: PASS"
    )
else:
    print(
        "CANONICAL EXTRACTION VALIDATION: REVIEW REQUIRED"
    )

print("=" * 75)

QWEN3-8B CLINICAL EXTRACTION POST-PROCESSING
Clinical facts: 20
Status/fact corrections: 4
Evidence quote repairs: 2
Exact grounded evidence: 20/20
Evidence grounding: 100.0 %
Validation errors: 0

ABSENT / NEGATED FACTS
---------------------------------------------------------------------------
Blood in vomit | there was no blood in your vomit
Blood in stool | No blood
Smoking | I don't smoke
Alcohol consumption | I don't drink alcohol

EVIDENCE REPAIRS
---------------------------------------------------------------------------
{
  "fact_number": 6,
  "fact": "Weakness and shakiness",
  "similarity": 0.949,
  "original_quote": "I've been feeling quite weak and shaky",
  "repaired_quote": "you've been feeling quite weak and shaky"
}
{
  "fact_number": 17,
  "fact": "Follow-up if symptoms persist",
  "similarity": 1.0,
  "original_quote": "if your symptoms haven't got better in three to four days",
  "repaired_quote": "If your symptoms haven't got better in three to four days,"
}

Saved

In [4]:
from pathlib import Path
import json
import re
import time
import torch
import pandas as pd
from difflib import SequenceMatcher


# ============================================================
# QWEN3-8B — FULL 10 CONSULTATION EXTRACTION
# ============================================================

print("=" * 78)
print("QWEN3-8B FULL 10-CONSULTATION CLINICAL EXTRACTION")
print("=" * 78)


# ------------------------------------------------------------
# Verify model is already loaded
# ------------------------------------------------------------

if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError(
        "Qwen3-8B is not loaded. Run the model-loading cell first."
    )


project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

transcript_dir = (
    project
    / "results/asr/whisper_large_v3/datalab_pilot"
)

output_root = (
    project
    / "results/nlp/qwen3_8b/full_pilot"
)

raw_dir = output_root / "raw"
extraction_dir = output_root / "extractions"
canonical_dir = output_root / "canonical"
metadata_dir = output_root / "metadata"

for folder in [
    raw_dir,
    extraction_dir,
    canonical_dir,
    metadata_dir
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


consultation_ids = [
    "day1_consultation01",
    "day1_consultation07",
    "day2_consultation01",
    "day2_consultation05",
    "day3_consultation06",
    "day3_consultation09",
    "day4_consultation03",
    "day4_consultation10",
    "day5_consultation03",
    "day5_consultation12"
]


# ============================================================
# FROZEN PROMPT
# ============================================================

def build_prompt(
    consultation_id,
    transcript
):

    return f"""
Extract the most clinically important facts from this
doctor-patient consultation.

Use ONLY information explicitly stated in the transcript.

STRICT REQUIREMENTS:

1. Return between 15 and 20 clinical facts TOTAL.
2. Do not duplicate the same clinical information.
3. Read the entire transcript, including the doctor's
   assessment, treatment plan and follow-up advice.
4. Prioritize clinically important information over ordinary
   social details.
5. Never invent or infer information.

CATEGORY DEFINITIONS:

presenting_complaint
- Main reason for consultation.

symptom
- Positive or explicitly denied symptom/finding.

temporal_detail
- Important onset, duration, frequency or severity.

medication
- Current medication only.

allergy
- Drug or substance allergy only.

medical_history
- Diagnosed past or current medical condition only.

exposure_history
- Relevant food, travel, infectious or environmental exposure.

social_history
- Smoking, alcohol, occupation or living situation only if
  clinically relevant.

assessment
- Doctor's working diagnosis or differential.

plan
- Treatment, medication advice, investigations or management.

safety_netting
- Follow-up instructions, return precautions or escalation advice.

STATUS RULE:

Use "present" when the clinical fact is affirmed.

Use "absent" when the patient explicitly denies the clinical fact.

Examples:

Fact: "Blood in vomit"
status: "absent"

Fact: "Smoking"
status: "absent"

Fact: "Asthma"
status: "present"

Do NOT write:
"No blood in vomit" with status "present".

EVIDENCE RULE:

Every evidence_quote MUST:
- be copied exactly from ONE continuous span of the transcript
- contain approximately 3 to 18 words
- not contain ellipses
- not combine separate parts of the transcript
- not paraphrase or correct the transcript

Do NOT use a doctor's question as evidence unless the patient's
answer confirming or denying the fact is included in the same
short continuous quote.

BALANCE THE OUTPUT:

Include:
- main complaint
- major symptoms
- clinically important negations
- important duration/frequency/numerical detail
- relevant history/medications/allergies/exposure
- assessment if stated
- important treatment plan
- follow-up or safety-netting

Do not spend most of the 15-20 facts only on symptoms.

Return JSON only.
No reasoning.
No explanation.
No Markdown.

Required structure:

{{
  "consultation_id": "{consultation_id}",
  "clinical_facts": [
    {{
      "category": "",
      "fact": "",
      "status": "present",
      "importance": "critical",
      "evidence_quote": ""
    }}
  ]
}}

importance must be exactly:
"critical" or "important"

TRANSCRIPT:

{transcript}
"""


# ============================================================
# HELPERS
# ============================================================

allowed_categories = {
    "presenting_complaint",
    "symptom",
    "temporal_detail",
    "medication",
    "allergy",
    "medical_history",
    "exposure_history",
    "social_history",
    "assessment",
    "plan",
    "safety_netting"
}

valid_status = {
    "present",
    "absent"
}

valid_importance = {
    "critical",
    "important"
}


def norm(text):

    text = str(text).lower()

    text = re.sub(
        r"[^\w\s]",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def repair_quote(
    quote,
    transcript
):

    # Already exact
    if quote in transcript:
        return (
            quote,
            1.0,
            False
        )

    transcript_words = transcript.split()

    quote_words = quote.split()

    target = norm(
        quote
    )

    n = len(
        quote_words
    )

    best_score = 0
    best_text = None

    min_size = max(
        3,
        n - 5
    )

    max_size = min(
        len(transcript_words),
        n + 6
    )

    for window_size in range(
        min_size,
        max_size + 1
    ):

        for start in range(
            0,
            len(transcript_words)
            - window_size
            + 1
        ):

            candidate = " ".join(
                transcript_words[
                    start:
                    start + window_size
                ]
            )

            score = SequenceMatcher(
                None,
                target,
                norm(candidate)
            ).ratio()

            if score > best_score:

                best_score = score
                best_text = candidate

    if best_score >= 0.78:

        return (
            best_text,
            best_score,
            True
        )

    return (
        quote,
        best_score,
        False
    )


def standardize_negative_fact(
    fact,
    status,
    evidence
):

    lower_fact = (
        str(fact)
        .lower()
        .strip()
    )

    lower_evidence = (
        str(evidence)
        .lower()
    )

    negative = False

    if lower_fact.startswith(
        "no "
    ):
        negative = True

    if lower_fact in {
        "non-smoker",
        "non smoker",
        "non-alcoholic",
        "non alcoholic"
    }:
        negative = True

    negative_evidence_patterns = [
        "no blood",
        "don't smoke",
        "do not smoke",
        "doesn't smoke",
        "does not smoke",
        "don't drink alcohol",
        "do not drink alcohol",
        "doesn't drink alcohol",
        "does not drink alcohol",
        "no vomiting",
        "no fever",
        "no pain",
        "no discharge",
        "no weakness",
        "no numbness",
        "no tingling",
        "no visual",
        "no chest pain",
        "no medication",
        "no medications"
    ]

    if any(
        pattern in lower_evidence
        for pattern
        in negative_evidence_patterns
    ):
        negative = True

    if negative:

        status = "absent"

        if lower_fact.startswith(
            "no "
        ):

            fact = fact[3:].strip()

            if fact:

                fact = (
                    fact[0].upper()
                    + fact[1:]
                )

        elif lower_fact in {
            "non-smoker",
            "non smoker"
        }:

            fact = "Smoking"

        elif lower_fact in {
            "non-alcoholic",
            "non alcoholic"
        }:

            fact = (
                "Alcohol consumption"
            )

    return (
        fact,
        status
    )


# ============================================================
# RUN ALL 10 CONSULTATIONS
# ============================================================

summary_rows = []

batch_start = time.time()


for number, consultation_id in enumerate(
    consultation_ids,
    start=1
):

    print("\n" + "=" * 78)

    print(
        f"[{number}/10] "
        f"{consultation_id}"
    )

    print("=" * 78)

    transcript_path = (
        transcript_dir
        / f"{consultation_id}_transcript.txt"
    )

    if not transcript_path.exists():

        print(
            "ERROR: transcript missing"
        )

        summary_rows.append({
            "consultation_id":
                consultation_id,

            "json_valid":
                False,

            "facts_extracted":
                0,

            "raw_evidence_grounding_percent":
                0,

            "canonical_evidence_grounding_percent":
                0,

            "status_corrections":
                0,

            "quote_repairs":
                0,

            "validation_errors":
                1,

            "generation_seconds":
                0,

            "peak_gpu_gb":
                0,

            "error":
                "transcript_missing"
        })

        continue


    transcript = (
        transcript_path
        .read_text(
            encoding="utf-8"
        )
    )

    prompt = build_prompt(
        consultation_id,
        transcript
    )

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = (
        tokenizer
        .apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(
        model.device
    )

    input_tokens = (
        inputs[
            "input_ids"
        ].shape[-1]
    )


    # --------------------------------------------------------
    # GENERATE
    # --------------------------------------------------------

    torch.cuda.reset_peak_memory_stats()

    start = time.time()

    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=2200,
            do_sample=False,
            pad_token_id=(
                tokenizer.eos_token_id
            )
        )

    generation_seconds = (
        time.time()
        - start
    )

    generated_ids = output_ids[
        0,
        input_tokens:
    ]

    generated_tokens = len(
        generated_ids
    )

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    peak_gpu_gb = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )


    # --------------------------------------------------------
    # SAVE RAW RESPONSE
    # --------------------------------------------------------

    raw_path = (
        raw_dir
        / f"{consultation_id}_qwen3_raw.txt"
    )

    raw_path.write_text(
        response,
        encoding="utf-8"
    )


    # --------------------------------------------------------
    # CLEAN
    # --------------------------------------------------------

    cleaned = response.strip()

    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned
    ).strip()

    first = cleaned.find(
        "{"
    )

    last = cleaned.rfind(
        "}"
    )

    if (
        first >= 0
        and last > first
    ):

        json_text = cleaned[
            first:
            last + 1
        ]

    else:

        json_text = cleaned


    # --------------------------------------------------------
    # PARSE
    # --------------------------------------------------------

    json_valid = False

    parsed = None

    json_error = None

    try:

        parsed = json.loads(
            json_text
        )

        json_valid = True

    except Exception as e:

        json_error = str(e)


    if not json_valid:

        print(
            "JSON VALID: False"
        )

        print(
            "Error:",
            json_error
        )

        summary_rows.append({
            "consultation_id":
                consultation_id,

            "json_valid":
                False,

            "facts_extracted":
                0,

            "raw_evidence_grounding_percent":
                0,

            "canonical_evidence_grounding_percent":
                0,

            "status_corrections":
                0,

            "quote_repairs":
                0,

            "validation_errors":
                1,

            "generation_seconds":
                round(
                    generation_seconds,
                    2
                ),

            "peak_gpu_gb":
                round(
                    peak_gpu_gb,
                    2
                ),

            "error":
                json_error
        })

        continue


    # --------------------------------------------------------
    # SAVE ORIGINAL EXTRACTION
    # --------------------------------------------------------

    extraction_path = (
        extraction_dir
        / (
            f"{consultation_id}"
            "_qwen3_extraction.json"
        )
    )

    with open(
        extraction_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            parsed,
            f,
            indent=2,
            ensure_ascii=False
        )


    facts = parsed.get(
        "clinical_facts",
        []
    )


    # --------------------------------------------------------
    # RAW GROUNDING BEFORE REPAIR
    # --------------------------------------------------------

    raw_exact = 0

    raw_total = 0

    for item in facts:

        quote = item.get(
            "evidence_quote",
            ""
        )

        if quote:

            raw_total += 1

            if quote in transcript:

                raw_exact += 1


    raw_grounding = (
        round(
            raw_exact
            / raw_total
            * 100,
            2
        )
        if raw_total
        else 0
    )


    # --------------------------------------------------------
    # POST-PROCESS
    # --------------------------------------------------------

    status_corrections = 0

    quote_repairs = 0

    repair_log = []


    for fact_number, item in enumerate(
        facts,
        start=1
    ):

        original_fact = item.get(
            "fact",
            ""
        )

        original_status = item.get(
            "status",
            ""
        )

        original_quote = item.get(
            "evidence_quote",
            ""
        )

        new_fact, new_status = (
            standardize_negative_fact(
                original_fact,
                original_status,
                original_quote
            )
        )

        if (
            new_fact != original_fact
            or new_status
            != original_status
        ):

            status_corrections += 1

        item["fact"] = (
            new_fact
        )

        item["status"] = (
            new_status
        )

        (
            repaired_quote,
            similarity,
            repaired
        ) = repair_quote(
            original_quote,
            transcript
        )

        if repaired:

            quote_repairs += 1

            item[
                "evidence_quote"
            ] = repaired_quote

            repair_log.append({
                "fact_number":
                    fact_number,

                "similarity":
                    round(
                        similarity,
                        3
                    ),

                "original_quote":
                    original_quote,

                "repaired_quote":
                    repaired_quote
            })


    # --------------------------------------------------------
    # FINAL VALIDATION
    # --------------------------------------------------------

    validation_errors = []

    canonical_exact = 0

    evidence_total = 0


    for fact_number, item in enumerate(
        facts,
        start=1
    ):

        category = item.get(
            "category",
            ""
        )

        status = item.get(
            "status",
            ""
        )

        importance = item.get(
            "importance",
            ""
        )

        quote = item.get(
            "evidence_quote",
            ""
        )

        if category not in allowed_categories:

            validation_errors.append(
                f"Fact {fact_number}: "
                "invalid category"
            )

        if status not in valid_status:

            validation_errors.append(
                f"Fact {fact_number}: "
                "invalid status"
            )

        if importance not in valid_importance:

            validation_errors.append(
                f"Fact {fact_number}: "
                "invalid importance"
            )

        if quote:

            evidence_total += 1

            if quote in transcript:

                canonical_exact += 1

            else:

                validation_errors.append(
                    f"Fact {fact_number}: "
                    "evidence not exact"
                )


    canonical_grounding = (
        round(
            canonical_exact
            / evidence_total
            * 100,
            2
        )
        if evidence_total
        else 0
    )


    # --------------------------------------------------------
    # SAVE CANONICAL
    # --------------------------------------------------------

    canonical = {
        "consultation_id":
            consultation_id,

        "model":
            "Qwen/Qwen3-8B",

        "thinking_enabled":
            False,

        "post_processing":
            True,

        "clinical_facts":
            facts
    }

    canonical_path = (
        canonical_dir
        / (
            f"{consultation_id}"
            "_qwen3_canonical.json"
        )
    )

    with open(
        canonical_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            canonical,
            f,
            indent=2,
            ensure_ascii=False
        )


    # --------------------------------------------------------
    # METADATA
    # --------------------------------------------------------

    metadata = {
        "consultation_id":
            consultation_id,

        "model":
            "Qwen/Qwen3-8B",

        "thinking_enabled":
            False,

        "input_tokens":
            input_tokens,

        "generated_tokens":
            generated_tokens,

        "hit_token_limit":
            generated_tokens >= 2200,

        "generation_seconds":
            round(
                generation_seconds,
                2
            ),

        "peak_gpu_gb":
            round(
                peak_gpu_gb,
                2
            ),

        "facts_extracted":
            len(facts),

        "raw_exact_evidence":
            raw_exact,

        "raw_evidence_total":
            raw_total,

        "raw_evidence_grounding_percent":
            raw_grounding,

        "status_corrections":
            status_corrections,

        "quote_repairs":
            quote_repairs,

        "canonical_exact_evidence":
            canonical_exact,

        "canonical_evidence_total":
            evidence_total,

        "canonical_evidence_grounding_percent":
            canonical_grounding,

        "validation_errors":
            validation_errors,

        "repair_log":
            repair_log
    }

    metadata_path = (
        metadata_dir
        / (
            f"{consultation_id}"
            "_qwen3_metadata.json"
        )
    )

    with open(
        metadata_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            metadata,
            f,
            indent=2,
            ensure_ascii=False
        )


    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    print(
        "JSON valid: True"
    )

    print(
        "Facts:",
        len(facts)
    )

    print(
        "Raw grounding:",
        raw_grounding,
        "%"
    )

    print(
        "Canonical grounding:",
        canonical_grounding,
        "%"
    )

    print(
        "Status corrections:",
        status_corrections
    )

    print(
        "Quote repairs:",
        quote_repairs
    )

    print(
        "Validation errors:",
        len(validation_errors)
    )

    print(
        "Generation:",
        round(
            generation_seconds,
            2
        ),
        "sec"
    )


    summary_rows.append({
        "consultation_id":
            consultation_id,

        "json_valid":
            True,

        "facts_extracted":
            len(facts),

        "target_15_20":
            (
                15 <= len(facts) <= 20
            ),

        "raw_evidence_grounding_percent":
            raw_grounding,

        "canonical_evidence_grounding_percent":
            canonical_grounding,

        "status_corrections":
            status_corrections,

        "quote_repairs":
            quote_repairs,

        "validation_errors":
            len(validation_errors),

        "generated_tokens":
            generated_tokens,

        "hit_token_limit":
            generated_tokens >= 2200,

        "generation_seconds":
            round(
                generation_seconds,
                2
            ),

        "peak_gpu_gb":
            round(
                peak_gpu_gb,
                2
            ),

        "error":
            None
    })


# ============================================================
# BATCH SUMMARY
# ============================================================

batch_seconds = (
    time.time()
    - batch_start
)

summary_df = pd.DataFrame(
    summary_rows
)

summary_csv = (
    output_root
    / "qwen3_8b_full_pilot_summary.csv"
)

summary_df.to_csv(
    summary_csv,
    index=False
)


summary_json = (
    output_root
    / "qwen3_8b_full_pilot_summary.json"
)

with open(
    summary_json,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "model":
                "Qwen/Qwen3-8B",

            "thinking_enabled":
                False,

            "consultations":
                10,

            "batch_seconds":
                round(
                    batch_seconds,
                    2
                ),

            "results":
                summary_df.to_dict(
                    orient="records"
                )
        },
        f,
        indent=2
    )


print("\n" + "=" * 78)
print("QWEN3-8B FULL PILOT SUMMARY")
print("=" * 78)

display_columns = [
    "consultation_id",
    "json_valid",
    "facts_extracted",
    "target_15_20",
    "raw_evidence_grounding_percent",
    "canonical_evidence_grounding_percent",
    "status_corrections",
    "quote_repairs",
    "validation_errors",
    "generation_seconds"
]

print(
    summary_df[
        display_columns
    ].to_string(
        index=False
    )
)

print("\nOVERALL")
print("-" * 78)

print(
    "Valid JSON:",
    f'{int(summary_df["json_valid"].sum())}/10'
)

print(
    "Target 15-20 facts:",
    f'{int(summary_df["target_15_20"].fillna(False).sum())}/10'
)

print(
    "Average facts:",
    round(
        summary_df[
            "facts_extracted"
        ].mean(),
        2
    )
)

print(
    "Average raw evidence grounding:",
    round(
        summary_df[
            "raw_evidence_grounding_percent"
        ].mean(),
        2
    ),
    "%"
)

print(
    "Average canonical evidence grounding:",
    round(
        summary_df[
            "canonical_evidence_grounding_percent"
        ].mean(),
        2
    ),
    "%"
)

print(
    "Total validation errors:",
    int(
        summary_df[
            "validation_errors"
        ].sum()
    )
)

print(
    "Total generation time:",
    round(
        summary_df[
            "generation_seconds"
        ].sum(),
        2
    ),
    "seconds"
)

print(
    "Full batch wall time:",
    round(
        batch_seconds,
        2
    ),
    "seconds"
)

print("\nSaved:")
print(summary_csv)
print(summary_json)

print("\n" + "=" * 78)
print("FULL QWEN EXTRACTION STAGE: COMPLETE")
print("=" * 78)


QWEN3-8B FULL 10-CONSULTATION CLINICAL EXTRACTION

[1/10] day1_consultation01
JSON valid: True
Facts: 20
Raw grounding: 90.0 %
Canonical grounding: 100.0 %
Status corrections: 4
Quote repairs: 2
Validation errors: 0
Generation: 25.6 sec

[2/10] day1_consultation07
JSON valid: True
Facts: 20
Raw grounding: 80.0 %
Canonical grounding: 100.0 %
Status corrections: 0
Quote repairs: 4
Validation errors: 0
Generation: 28.9 sec

[3/10] day2_consultation01
JSON valid: True
Facts: 20
Raw grounding: 75.0 %
Canonical grounding: 100.0 %
Status corrections: 0
Quote repairs: 5
Validation errors: 0
Generation: 30.02 sec

[4/10] day2_consultation05
JSON valid: True
Facts: 20
Raw grounding: 70.0 %
Canonical grounding: 100.0 %
Status corrections: 0
Quote repairs: 6
Validation errors: 0
Generation: 32.45 sec

[5/10] day3_consultation06
JSON valid: True
Facts: 19
Raw grounding: 63.16 %
Canonical grounding: 89.47 %
Status corrections: 2
Quote repairs: 5
Validation errors: 2
Generation: 24.13 sec

[6/10] day

In [5]:
from pathlib import Path
import json
import re
from difflib import SequenceMatcher

print("=" * 78)
print("UNRESOLVED QWEN EVIDENCE VALIDATION ERRORS")
print("=" * 78)

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

canonical_dir = (
    project
    / "results/nlp/qwen3_8b/full_pilot/canonical"
)

transcript_dir = (
    project
    / "results/asr/whisper_large_v3/datalab_pilot"
)

problem_ids = [
    "day3_consultation06",
    "day4_consultation10",
    "day5_consultation03"
]


def norm(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def best_transcript_match(
    quote,
    transcript
):

    transcript_words = transcript.split()
    quote_words = quote.split()

    target = norm(quote)

    n = len(quote_words)

    best_score = 0
    best_text = ""

    for window_size in range(
        max(3, n - 6),
        min(
            len(transcript_words),
            n + 7
        ) + 1
    ):

        for start in range(
            0,
            len(transcript_words)
            - window_size
            + 1
        ):

            candidate = " ".join(
                transcript_words[
                    start:
                    start + window_size
                ]
            )

            score = SequenceMatcher(
                None,
                target,
                norm(candidate)
            ).ratio()

            if score > best_score:

                best_score = score
                best_text = candidate

    return best_score, best_text


total_unresolved = 0


for consultation_id in problem_ids:

    json_path = (
        canonical_dir
        / f"{consultation_id}_qwen3_canonical.json"
    )

    transcript_path = (
        transcript_dir
        / f"{consultation_id}_transcript.txt"
    )

    with open(
        json_path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    transcript = transcript_path.read_text(
        encoding="utf-8"
    )

    print("\n" + "=" * 78)
    print(consultation_id)
    print("=" * 78)

    consultation_errors = 0

    for number, item in enumerate(
        data.get(
            "clinical_facts",
            []
        ),
        start=1
    ):

        quote = item.get(
            "evidence_quote",
            ""
        )

        if quote not in transcript:

            consultation_errors += 1
            total_unresolved += 1

            score, best_match = (
                best_transcript_match(
                    quote,
                    transcript
                )
            )

            print(
                f"\nFact #{number}"
            )

            print(
                "Category:",
                item.get("category")
            )

            print(
                "Fact:",
                item.get("fact")
            )

            print(
                "Status:",
                item.get("status")
            )

            print(
                "Importance:",
                item.get("importance")
            )

            print(
                "Generated evidence:"
            )

            print(
                quote
            )

            print(
                "\nClosest transcript span:"
            )

            print(
                best_match
            )

            print(
                "\nSimilarity:",
                round(
                    score,
                    3
                )
            )

            print("-" * 78)

    print(
        "\nUnresolved in consultation:",
        consultation_errors
    )


print("\n" + "=" * 78)
print(
    "TOTAL UNRESOLVED:",
    total_unresolved
)
print("=" * 78)


UNRESOLVED QWEN EVIDENCE VALIDATION ERRORS

day3_consultation06

Fact #7
Category: medical_history
Fact: Eczema
Status: present
Importance: important
Generated evidence:
you have eczema

Closest transcript span:
uh a eczema

Similarity: 0.769
------------------------------------------------------------------------------

Fact #11
Category: assessment
Fact: Anaphylactic reaction concern
Status: present
Importance: critical
Generated evidence:
we are worried about anaphylactic reaction

Closest transcript span:
an anaphylactic reaction

Similarity: 0.697
------------------------------------------------------------------------------

Unresolved in consultation: 2

day4_consultation10

Fact #5
Category: temporal_detail
Fact: Duration of symptoms
Status: present
Importance: critical
Generated evidence:
I've been feeling sick in my lower tummy for the last couple of days

Closest transcript span:
you've been feeling sick in your lower tummy yeah it's more of an

Similarity: 0.707
-----------

In [6]:
from pathlib import Path
import json
import pandas as pd

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

output_dir = (
    project
    / "results/nlp/qwen3_8b/evaluation"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

rows = [
    {
        "consultation_id": "day3_consultation06",
        "fact_number": 7,
        "fact": "Eczema",
        "automatic_evidence_match": False,
        "manual_fact_supported": True,
        "classification": "evidence_quote_mismatch",
        "hallucination": False,
        "reason": "Eczema is explicitly present in the transcript as 'uh a eczema'."
    },
    {
        "consultation_id": "day3_consultation06",
        "fact_number": 11,
        "fact": "Anaphylactic reaction concern",
        "automatic_evidence_match": False,
        "manual_fact_supported": True,
        "classification": "evidence_quote_mismatch",
        "hallucination": False,
        "reason": "The transcript explicitly states 'an anaphylactic reaction'."
    },
    {
        "consultation_id": "day4_consultation10",
        "fact_number": 5,
        "fact": "Duration of symptoms",
        "automatic_evidence_match": False,
        "manual_fact_supported": True,
        "classification": "evidence_quote_mismatch",
        "hallucination": False,
        "reason": "The consultation explicitly states that the abdominal symptoms were present for a couple of days."
    },
    {
        "consultation_id": "day4_consultation10",
        "fact_number": 6,
        "fact": "Constipation",
        "automatic_evidence_match": False,
        "manual_fact_supported": True,
        "classification": "evidence_quote_mismatch",
        "hallucination": False,
        "reason": "Constipation about one week earlier is explicitly stated; the generated quote removed filler words."
    },
    {
        "consultation_id": "day5_consultation03",
        "fact_number": 4,
        "fact": "Sleep disturbance",
        "automatic_evidence_match": False,
        "manual_fact_supported": True,
        "classification": "evidence_quote_mismatch",
        "hallucination": False,
        "reason": "Difficulty getting to sleep is explicitly present in the transcript."
    }
]

df = pd.DataFrame(rows)

csv_path = (
    output_dir
    / "qwen3_manual_evidence_adjudication.csv"
)

json_path = (
    output_dir
    / "qwen3_manual_evidence_adjudication.json"
)

df.to_csv(
    csv_path,
    index=False
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "total_flagged_items": 5,
            "supported_after_manual_review": 5,
            "unsupported_after_manual_review": 0,
            "hallucinations_confirmed": 0,
            "note": (
                "These five items failed exact automatic evidence matching "
                "because the generated evidence quote paraphrased or cleaned "
                "the transcript wording. Manual review confirmed that the "
                "underlying clinical facts were supported."
            ),
            "items": rows
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 75)
print("QWEN MANUAL EVIDENCE ADJUDICATION")
print("=" * 75)

print("Flagged items:", len(df))
print(
    "Supported after manual review:",
    int(df["manual_fact_supported"].sum())
)
print(
    "Unsupported after manual review:",
    int((~df["manual_fact_supported"]).sum())
)
print(
    "Confirmed hallucinations:",
    int(df["hallucination"].sum())
)

print("\nSaved:")
print(csv_path)
print(json_path)

print("\nMANUAL ADJUDICATION: COMPLETE")

QWEN MANUAL EVIDENCE ADJUDICATION
Flagged items: 5
Supported after manual review: 5
Unsupported after manual review: 0
Confirmed hallucinations: 0

Saved:
/home/jovyan/Case_Study_2_Medical_Consultation_AI/results/nlp/qwen3_8b/evaluation/qwen3_manual_evidence_adjudication.csv
/home/jovyan/Case_Study_2_Medical_Consultation_AI/results/nlp/qwen3_8b/evaluation/qwen3_manual_evidence_adjudication.json

MANUAL ADJUDICATION: COMPLETE


In [7]:
from pathlib import Path
import pandas as pd
import json
import re
import unicodedata
from difflib import SequenceMatcher


print("=" * 78)
print("QWEN3-8B GOLD CLINICAL FACT COVERAGE SCREENING")
print("=" * 78)


# ============================================================
# PATHS
# ============================================================

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

gold_path = (
    project
    / "data/PriMock57_10_Pilot_Clinical_Gold_Checklist.csv"
)

canonical_dir = (
    project
    / "results/nlp/qwen3_8b/full_pilot/canonical"
)

output_dir = (
    project
    / "results/nlp/qwen3_8b/evaluation"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_text(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    ).lower()

    cleaned = []

    for character in text:

        category = unicodedata.category(
            character
        )

        if (
            category.startswith("P")
            or category.startswith("S")
        ):
            cleaned.append(" ")
        else:
            cleaned.append(character)

    text = "".join(cleaned)

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


# ============================================================
# LOAD GOLD
# ============================================================

gold_df = pd.read_csv(
    gold_path
)

print(
    "Gold facts:",
    len(gold_df)
)

print(
    "Consultations:",
    gold_df[
        "consultation_id"
    ].nunique()
)


# ============================================================
# LOAD QWEN CANONICAL EXTRACTIONS
# ============================================================

qwen_by_consultation = {}

for path in sorted(
    canonical_dir.glob(
        "*_qwen3_canonical.json"
    )
):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        data = json.load(f)

    cid = data[
        "consultation_id"
    ]

    qwen_by_consultation[
        cid
    ] = data.get(
        "clinical_facts",
        []
    )


print(
    "Qwen canonical files:",
    len(qwen_by_consultation)
)

if len(qwen_by_consultation) != 10:
    raise RuntimeError(
        "Expected 10 canonical Qwen extractions."
    )


# ============================================================
# MATCH GOLD FACTS
# ============================================================

results = []


for _, gold in gold_df.iterrows():

    cid = gold[
        "consultation_id"
    ]

    expected_fact = str(
        gold[
            "expected_fact"
        ]
    )

    gold_polarity = str(
        gold[
            "polarity"
        ]
    ).lower()

    patterns = [
        p.strip()
        for p in str(
            gold[
                "acceptable_patterns"
            ]
        ).split("||")
        if p.strip()
    ]

    qwen_facts = (
        qwen_by_consultation.get(
            cid,
            []
        )
    )

    pattern_match = False
    polarity_match = False

    matched_number = None
    matched_fact = None
    matched_status = None
    matched_quote = None
    matched_pattern = None

    # --------------------------------------------------------
    # FIRST: strict pattern + polarity match
    # --------------------------------------------------------

    for number, item in enumerate(
        qwen_facts,
        start=1
    ):

        fact = str(
            item.get(
                "fact",
                ""
            )
        )

        quote = str(
            item.get(
                "evidence_quote",
                ""
            )
        )

        status = str(
            item.get(
                "status",
                ""
            )
        ).lower()

        combined = normalize_text(
            fact + " " + quote
        )

        for pattern in patterns:

            try:

                match = re.search(
                    pattern,
                    combined,
                    flags=re.IGNORECASE
                )

            except re.error:
                match = None

            if match:

                pattern_match = True

                if status == gold_polarity:

                    polarity_match = True

                    matched_number = number
                    matched_fact = fact
                    matched_status = status
                    matched_quote = quote
                    matched_pattern = pattern

                    break

        if polarity_match:
            break


    # --------------------------------------------------------
    # SECOND: find best semantic/lexical candidate for review
    # --------------------------------------------------------

    best_similarity = 0
    best_candidate_number = None
    best_candidate_fact = None
    best_candidate_status = None
    best_candidate_quote = None

    expected_normalized = normalize_text(
        expected_fact
    )

    for number, item in enumerate(
        qwen_facts,
        start=1
    ):

        candidate_fact = str(
            item.get(
                "fact",
                ""
            )
        )

        candidate_quote = str(
            item.get(
                "evidence_quote",
                ""
            )
        )

        candidate_status = str(
            item.get(
                "status",
                ""
            )
        ).lower()

        score = SequenceMatcher(
            None,
            expected_normalized,
            normalize_text(
                candidate_fact
            )
        ).ratio()

        if score > best_similarity:

            best_similarity = score

            best_candidate_number = (
                number
            )

            best_candidate_fact = (
                candidate_fact
            )

            best_candidate_status = (
                candidate_status
            )

            best_candidate_quote = (
                candidate_quote
            )


    automatic_detected = (
        pattern_match
        and polarity_match
    )

    results.append({
        "consultation_id":
            cid,

        "gold_category":
            gold["category"],

        "gold_concept":
            gold["concept"],

        "expected_fact":
            expected_fact,

        "criticality":
            gold["criticality"],

        "gold_polarity":
            gold_polarity,

        "automatic_detected":
            automatic_detected,

        "pattern_matched":
            pattern_match,

        "polarity_matched":
            polarity_match,

        "matched_qwen_fact_number":
            matched_number,

        "matched_qwen_fact":
            matched_fact,

        "matched_qwen_status":
            matched_status,

        "matched_evidence_quote":
            matched_quote,

        "matched_pattern":
            matched_pattern,

        "best_candidate_number":
            best_candidate_number,

        "best_candidate_fact":
            best_candidate_fact,

        "best_candidate_status":
            best_candidate_status,

        "best_candidate_quote":
            best_candidate_quote,

        "best_candidate_similarity":
            round(
                best_similarity,
                3
            ),

        "manual_review_required":
            not automatic_detected
    })


results_df = pd.DataFrame(
    results
)


# ============================================================
# SUMMARY
# ============================================================

total = len(
    results_df
)

detected = int(
    results_df[
        "automatic_detected"
    ].sum()
)

review_required = int(
    results_df[
        "manual_review_required"
    ].sum()
)


critical_df = results_df[
    results_df[
        "criticality"
    ] == "Critical"
]

critical_detected = int(
    critical_df[
        "automatic_detected"
    ].sum()
)


negative_df = results_df[
    results_df[
        "gold_polarity"
    ] == "absent"
]

negative_detected = int(
    negative_df[
        "automatic_detected"
    ].sum()
)


print("\n" + "=" * 78)
print("AUTOMATIC GOLD COVERAGE SCREENING SUMMARY")
print("=" * 78)

print(
    "Gold facts:",
    total
)

print(
    "Automatically matched:",
    detected
)

print(
    "Automatic screening recall:",
    round(
        detected
        / total
        * 100,
        2
    ),
    "%"
)

print(
    "Manual review required:",
    review_required
)

print(
    "\nCritical facts automatically matched:",
    f"{critical_detected}/{len(critical_df)}"
)

print(
    "Critical automatic recall:",
    round(
        critical_detected
        / len(critical_df)
        * 100,
        2
    ),
    "%"
)

print(
    "\nNegated facts automatically matched:",
    f"{negative_detected}/{len(negative_df)}"
)

print(
    "Negation automatic recall:",
    round(
        negative_detected
        / len(negative_df)
        * 100,
        2
    ),
    "%"
)


# ============================================================
# PER CONSULTATION
# ============================================================

consultation_summary = (
    results_df
    .groupby(
        "consultation_id"
    )
    .agg(
        gold_facts=(
            "automatic_detected",
            "size"
        ),

        automatically_matched=(
            "automatic_detected",
            "sum"
        ),

        manual_review_required=(
            "manual_review_required",
            "sum"
        )
    )
    .reset_index()
)

consultation_summary[
    "automatic_recall_percent"
] = (
    consultation_summary[
        "automatically_matched"
    ]
    / consultation_summary[
        "gold_facts"
    ]
    * 100
).round(2)


print("\nPER-CONSULTATION SCREENING")
print("-" * 78)

print(
    consultation_summary
    .to_string(
        index=False
    )
)


# ============================================================
# SAVE
# ============================================================

detailed_path = (
    output_dir
    / "qwen3_gold_fact_coverage_screening.csv"
)

summary_path = (
    output_dir
    / "qwen3_gold_fact_coverage_by_consultation.csv"
)

review_path = (
    output_dir
    / "qwen3_gold_fact_manual_review_required.csv"
)

results_df.to_csv(
    detailed_path,
    index=False
)

consultation_summary.to_csv(
    summary_path,
    index=False
)

results_df[
    results_df[
        "manual_review_required"
    ]
].to_csv(
    review_path,
    index=False
)


print("\nSaved:")
print(detailed_path)
print(summary_path)
print(review_path)

print("\n" + "=" * 78)
print("GOLD COVERAGE SCREENING: COMPLETE")
print("=" * 78)

QWEN3-8B GOLD CLINICAL FACT COVERAGE SCREENING
Gold facts: 150
Consultations: 10
Qwen canonical files: 10

AUTOMATIC GOLD COVERAGE SCREENING SUMMARY
Gold facts: 150
Automatically matched: 107
Automatic screening recall: 71.33 %
Manual review required: 43

Critical facts automatically matched: 65/85
Critical automatic recall: 76.47 %

Negated facts automatically matched: 6/25
Negation automatic recall: 24.0 %

PER-CONSULTATION SCREENING
------------------------------------------------------------------------------
    consultation_id  gold_facts  automatically_matched  manual_review_required  automatic_recall_percent
day1_consultation01          15                     12                       3                     80.00
day1_consultation07          15                     10                       5                     66.67
day2_consultation01          15                     11                       4                     73.33
day2_consultation05          15                     12       

In [8]:
from pathlib import Path
import pandas as pd
import json
import re
from difflib import SequenceMatcher

print("=" * 78)
print("BUILDING QWEN GOLD-FACT MANUAL REVIEW PACKET")
print("=" * 78)

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

screening_path = (
    project
    / "results/nlp/qwen3_8b/evaluation"
    / "qwen3_gold_fact_coverage_screening.csv"
)

canonical_dir = (
    project
    / "results/nlp/qwen3_8b/full_pilot/canonical"
)

output_dir = (
    project
    / "results/nlp/qwen3_8b/evaluation"
)

screening = pd.read_csv(
    screening_path
)

review = screening[
    screening["manual_review_required"] == True
].copy()

print("Cases requiring review:", len(review))


def norm(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


rows = []


for _, gold in review.iterrows():

    cid = gold["consultation_id"]

    canonical_path = (
        canonical_dir
        / f"{cid}_qwen3_canonical.json"
    )

    with open(
        canonical_path,
        "r",
        encoding="utf-8"
    ) as f:
        data = json.load(f)

    qwen_facts = data.get(
        "clinical_facts",
        []
    )

    expected = norm(
        gold["expected_fact"]
    )

    candidates = []

    for number, item in enumerate(
        qwen_facts,
        start=1
    ):

        candidate_fact = item.get(
            "fact",
            ""
        )

        candidate_quote = item.get(
            "evidence_quote",
            ""
        )

        score_fact = SequenceMatcher(
            None,
            expected,
            norm(candidate_fact)
        ).ratio()

        score_quote = SequenceMatcher(
            None,
            expected,
            norm(candidate_quote)
        ).ratio()

        score = max(
            score_fact,
            score_quote
        )

        candidates.append({
            "number": number,
            "fact": candidate_fact,
            "status": item.get(
                "status",
                ""
            ),
            "category": item.get(
                "category",
                ""
            ),
            "importance": item.get(
                "importance",
                ""
            ),
            "quote": candidate_quote,
            "similarity": round(
                score,
                3
            )
        })

    candidates = sorted(
        candidates,
        key=lambda x: x["similarity"],
        reverse=True
    )[:5]

    row = {
        "consultation_id":
            cid,

        "gold_category":
            gold["gold_category"],

        "gold_concept":
            gold["gold_concept"],

        "expected_fact":
            gold["expected_fact"],

        "criticality":
            gold["criticality"],

        "gold_polarity":
            gold["gold_polarity"]
    }

    for rank in range(5):

        prefix = f"candidate_{rank + 1}"

        if rank < len(candidates):

            c = candidates[rank]

            row[
                f"{prefix}_number"
            ] = c["number"]

            row[
                f"{prefix}_fact"
            ] = c["fact"]

            row[
                f"{prefix}_status"
            ] = c["status"]

            row[
                f"{prefix}_category"
            ] = c["category"]

            row[
                f"{prefix}_quote"
            ] = c["quote"]

            row[
                f"{prefix}_similarity"
            ] = c["similarity"]

        else:

            row[
                f"{prefix}_number"
            ] = ""

            row[
                f"{prefix}_fact"
            ] = ""

            row[
                f"{prefix}_status"
            ] = ""

            row[
                f"{prefix}_category"
            ] = ""

            row[
                f"{prefix}_quote"
            ] = ""

            row[
                f"{prefix}_similarity"
            ] = ""

    rows.append(row)


review_packet = pd.DataFrame(
    rows
)

csv_path = (
    output_dir
    / "qwen3_gold_fact_manual_review_packet.csv"
)

review_packet.to_csv(
    csv_path,
    index=False
)

print("\nManual-review rows:", len(review_packet))

print(
    "Critical rows:",
    int(
        (
            review_packet[
                "criticality"
            ]
            == "Critical"
        ).sum()
    )
)

print(
    "Negated rows:",
    int(
        (
            review_packet[
                "gold_polarity"
            ]
            == "absent"
        ).sum()
    )
)

print("\nSaved:")
print(csv_path)

print("\n" + "=" * 78)
print("MANUAL REVIEW PACKET: READY")
print("=" * 78)

BUILDING QWEN GOLD-FACT MANUAL REVIEW PACKET
Cases requiring review: 43

Manual-review rows: 43
Critical rows: 20
Negated rows: 19

Saved:
/home/jovyan/Case_Study_2_Medical_Consultation_AI/results/nlp/qwen3_8b/evaluation/qwen3_gold_fact_manual_review_packet.csv

MANUAL REVIEW PACKET: READY


In [9]:
from pathlib import Path
import pandas as pd
import json

print("=" * 78)
print("QWEN3-8B FINAL GOLD-FACT MANUAL ADJUDICATION")
print("=" * 78)

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

review_path = (
    project
    / "results/nlp/qwen3_8b/evaluation"
    / "qwen3_gold_fact_manual_review_packet.csv"
)

screening_path = (
    project
    / "results/nlp/qwen3_8b/evaluation"
    / "qwen3_gold_fact_coverage_screening.csv"
)

output_dir = (
    project
    / "results/nlp/qwen3_8b/evaluation"
)

review = pd.read_csv(review_path)
screening = pd.read_csv(screening_path)

# ------------------------------------------------------------
# Manual adjudication
# Row numbers correspond to the 43-row review packet.
# ------------------------------------------------------------

manual_hits = {
    9: (
        "Gold fact captured: ear discharge is represented "
        "as absent."
    ),

    10: (
        "Gold fact preserved in the extracted dizziness item; "
        "the evidence explicitly states that it is not proper "
        "room-spinning vertigo."
    ),

    12: (
        "Gold fact captured explicitly as follow-up in "
        "four to five days."
    ),

    16: (
        "Gold fact captured: prior similar reaction is "
        "represented as absent."
    ),

    18: (
        "Antihistamine administration and the ambulance plan "
        "were both extracted, preserving the emergency action."
    ),

    19: (
        "Immediate emergency-room/hospital assessment "
        "was captured."
    ),

    28: (
        "The plan explicitly captured avoidance of codeine. "
        "The gold polarity represents a negative instruction."
    ),

    40: (
        "Nausea and vomiting were both extracted as absent."
    )
}


special_miss_reasons = {

    1: (
        "Frequent bowel movements were extracted, but the "
        "specific 6-7 times/day numerical detail was omitted."
    ),

    2: (
        "Lower abdominal pain was extracted, but left-sided "
        "laterality was omitted."
    ),

    3: (
        "Vomiting at symptom onset was extracted, but the "
        "fact that vomiting had stopped was omitted."
    ),

    4: (
        "Cough/cold information was present, but the specific "
        "dry cough with morning phlegm detail was omitted."
    ),

    7: (
        "Muscle/joint discomfort was extracted, but the "
        "specific elbow/knee stiffness fact was not preserved."
    ),

    29: (
        "The model interpreted increased toilet frequency as "
        "bowel movements rather than urinary frequency."
    ),

    34: (
        "Urgent worsening advice was captured, but the "
        "same-day face-to-face assessment component was omitted."
    ),

    35: (
        "Sleep disturbance was captured, but the clinically "
        "important 3-4 hours per night detail was omitted."
    )
}


rows = []

for idx, row in review.reset_index(
    drop=True
).iterrows():

    review_number = idx + 1

    is_hit = (
        review_number
        in manual_hits
    )

    if is_hit:

        reason = manual_hits[
            review_number
        ]

        classification = (
            "true_extraction_hit"
        )

    else:

        reason = (
            special_miss_reasons.get(
                review_number,
                (
                    "No Qwen candidate preserved the full "
                    "clinically important gold fact."
                )
            )
        )

        classification = (
            "genuine_omission"
        )

    rows.append({
        "review_number":
            review_number,

        "consultation_id":
            row["consultation_id"],

        "gold_category":
            row["gold_category"],

        "gold_concept":
            row["gold_concept"],

        "expected_fact":
            row["expected_fact"],

        "criticality":
            row["criticality"],

        "gold_polarity":
            row["gold_polarity"],

        "manual_detected":
            is_hit,

        "classification":
            classification,

        "adjudication_reason":
            reason
    })


adjudication_df = pd.DataFrame(
    rows
)


# ------------------------------------------------------------
# Merge manual decisions into complete 150-fact screening
# ------------------------------------------------------------

final_df = screening.copy()

final_df[
    "manual_detected"
] = False

final_df[
    "final_detected"
] = final_df[
    "automatic_detected"
].astype(bool)

final_df[
    "adjudication_method"
] = "automatic_pattern_match"

final_df[
    "manual_adjudication_reason"
] = ""


for _, adjudicated in (
    adjudication_df.iterrows()
):

    mask = (
        (
            final_df[
                "consultation_id"
            ]
            == adjudicated[
                "consultation_id"
            ]
        )
        &
        (
            final_df[
                "gold_concept"
            ]
            == adjudicated[
                "gold_concept"
            ]
        )
    )

    final_df.loc[
        mask,
        "manual_detected"
    ] = adjudicated[
        "manual_detected"
    ]

    final_df.loc[
        mask,
        "final_detected"
    ] = adjudicated[
        "manual_detected"
    ]

    final_df.loc[
        mask,
        "adjudication_method"
    ] = "manual_review"

    final_df.loc[
        mask,
        "manual_adjudication_reason"
    ] = adjudicated[
        "adjudication_reason"
    ]


# ------------------------------------------------------------
# Final metrics
# ------------------------------------------------------------

total = len(final_df)

final_hits = int(
    final_df[
        "final_detected"
    ].sum()
)


critical = final_df[
    final_df[
        "criticality"
    ] == "Critical"
]

critical_hits = int(
    critical[
        "final_detected"
    ].sum()
)


negations = final_df[
    final_df[
        "gold_polarity"
    ] == "absent"
]

negation_hits = int(
    negations[
        "final_detected"
    ].sum()
)


final_recall = round(
    final_hits
    / total
    * 100,
    2
)

critical_recall = round(
    critical_hits
    / len(critical)
    * 100,
    2
)

negation_recall = round(
    negation_hits
    / len(negations)
    * 100,
    2
)


# ------------------------------------------------------------
# Per-consultation final metrics
# ------------------------------------------------------------

consultation_summary = (
    final_df
    .groupby(
        "consultation_id"
    )
    .agg(
        gold_facts=(
            "final_detected",
            "size"
        ),

        final_hits=(
            "final_detected",
            "sum"
        )
    )
    .reset_index()
)

consultation_summary[
    "final_recall_percent"
] = (
    consultation_summary[
        "final_hits"
    ]
    / consultation_summary[
        "gold_facts"
    ]
    * 100
).round(2)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

adjudication_csv = (
    output_dir
    / "qwen3_gold_fact_manual_adjudication.csv"
)

final_csv = (
    output_dir
    / "qwen3_gold_fact_final_evaluation.csv"
)

consultation_csv = (
    output_dir
    / "qwen3_gold_fact_final_by_consultation.csv"
)

summary_json = (
    output_dir
    / "qwen3_gold_fact_final_summary.json"
)


adjudication_df.to_csv(
    adjudication_csv,
    index=False
)

final_df.to_csv(
    final_csv,
    index=False
)

consultation_summary.to_csv(
    consultation_csv,
    index=False
)


summary = {
    "model":
        "Qwen/Qwen3-8B",

    "gold_facts":
        total,

    "automatic_matches":
        107,

    "manual_review_items":
        43,

    "additional_manual_hits":
        int(
            adjudication_df[
                "manual_detected"
            ].sum()
        ),

    "genuine_omissions_after_review":
        int(
            (
                ~adjudication_df[
                    "manual_detected"
                ]
            ).sum()
        ),

    "final_detected_facts":
        final_hits,

    "overall_clinical_fact_recall_percent":
        final_recall,

    "critical_facts_total":
        len(critical),

    "critical_facts_detected":
        critical_hits,

    "critical_fact_recall_percent":
        critical_recall,

    "negated_facts_total":
        len(negations),

    "negated_facts_detected":
        negation_hits,

    "negation_recall_percent":
        negation_recall,

    "note": (
        "Precision/F1 are not calculated against this checklist "
        "because the 150-item gold checklist is a compact set of "
        "clinically salient facts rather than an exhaustive "
        "annotation of every valid fact in each consultation."
    )
}


with open(
    summary_json,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print(
    "Manual-review items:",
    len(adjudication_df)
)

print(
    "Additional true hits:",
    int(
        adjudication_df[
            "manual_detected"
        ].sum()
    )
)

print(
    "Genuine omissions:",
    int(
        (
            ~adjudication_df[
                "manual_detected"
            ]
        ).sum()
    )
)

print("\n" + "=" * 78)
print("FINAL QWEN CLINICAL EXTRACTION METRICS")
print("=" * 78)

print(
    "Overall clinical fact recall:",
    f"{final_hits}/{total}",
    "=",
    final_recall,
    "%"
)

print(
    "Critical fact recall:",
    f"{critical_hits}/{len(critical)}",
    "=",
    critical_recall,
    "%"
)

print(
    "Negation recall:",
    f"{negation_hits}/{len(negations)}",
    "=",
    negation_recall,
    "%"
)

print("\nPER-CONSULTATION FINAL RECALL")
print("-" * 78)

print(
    consultation_summary.to_string(
        index=False
    )
)

print("\nSaved:")
print(adjudication_csv)
print(final_csv)
print(consultation_csv)
print(summary_json)

print("\n" + "=" * 78)
print("FINAL GOLD-FACT EVALUATION: COMPLETE")
print("=" * 78)

QWEN3-8B FINAL GOLD-FACT MANUAL ADJUDICATION
Manual-review items: 43
Additional true hits: 8
Genuine omissions: 35

FINAL QWEN CLINICAL EXTRACTION METRICS
Overall clinical fact recall: 115/150 = 76.67 %
Critical fact recall: 69/85 = 81.18 %
Negation recall: 11/25 = 44.0 %

PER-CONSULTATION FINAL RECALL
------------------------------------------------------------------------------
    consultation_id  gold_facts  final_hits  final_recall_percent
day1_consultation01          15          12                 80.00
day1_consultation07          15          10                 66.67
day2_consultation01          15          14                 93.33
day2_consultation05          15          12                 80.00
day3_consultation06          15          14                 93.33
day3_consultation09          15           9                 60.00
day4_consultation03          15          13                 86.67
day4_consultation10          15           9                 60.00
day5_consultation03    

In [10]:
from pathlib import Path
import json
import zipfile
import pandas as pd

print("=" * 78)
print("FINAL NLP EXTRACTION STAGE FREEZE")
print("=" * 78)

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

final_dir = (
    project
    / "results/nlp/final_selection"
)

final_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# FINAL MODEL SELECTION
# ============================================================

selection = {
    "selected_model": "Qwen/Qwen3-8B",
    "selection_status": "FINAL",
    "pipeline": (
        "Whisper Large-v3 transcript -> "
        "Qwen3-8B clinical fact extraction -> "
        "deterministic validation/post-processing -> "
        "canonical clinical JSON"
    ),
    "qwen_configuration": {
        "thinking_enabled": False,
        "structured_output": "JSON",
        "target_facts_per_consultation": "15-20"
    },
    "full_pilot_results": {
        "consultations": 10,
        "valid_json": "10/10",
        "average_facts_extracted": 19.8,
        "average_raw_evidence_grounding_percent": 68.92,
        "average_canonical_evidence_grounding_percent": 97.39,
        "total_generation_seconds": 291.78,
        "full_batch_wall_seconds": 414.56
    },
    "gold_fact_evaluation": {
        "gold_facts": 150,
        "final_detected_facts": 115,
        "overall_clinical_fact_recall_percent": 76.67,
        "critical_facts": 85,
        "critical_facts_detected": 69,
        "critical_fact_recall_percent": 81.18,
        "negated_facts": 25,
        "negated_facts_detected": 11,
        "negation_recall_percent": 44.0
    },
    "evidence_adjudication": {
        "automatically_flagged_items": 5,
        "supported_after_manual_review": 5,
        "confirmed_unsupported_among_flagged": 0,
        "important_note": (
            "This does not establish a global zero hallucination rate. "
            "It only means all five automatically unresolved evidence "
            "cases were manually confirmed as supported."
        )
    },
    "medgemma_comparison": {
        "model": "google/medgemma-1.5-4b-it",
        "successful_compact_test_json_valid": True,
        "successful_compact_test_facts": 43,
        "successful_compact_test_evidence_grounding_percent": 90.7,
        "successful_compact_test_generation_seconds": 133.18,
        "observed_issue": (
            "Structured generation was less stable and more verbose; "
            "other prompt variants reached output-token limits or "
            "introduced categorization problems."
        )
    },
    "selection_reason": (
        "Qwen3-8B was selected because it produced stable valid JSON, "
        "was substantially faster in compatibility testing, supported "
        "thinking-disabled generation, and integrated cleanly with the "
        "deterministic grounding and validation layer."
    ),
    "important_limitation": (
        "Negation recall was 44%, so human review remains necessary "
        "before approval of the final medical record."
    ),
    "next_stage": (
        "Generate structured SOAP medical records from canonical "
        "clinical JSON and implement human-in-the-loop review."
    )
}

selection_json = (
    final_dir
    / "final_nlp_model_selection.json"
)

with open(
    selection_json,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        selection,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# SMALL FINAL COMPARISON TABLE
# ============================================================

comparison = pd.DataFrame([
    {
        "model": "Qwen3-8B",
        "structured_json_success": "10/10 full pilot",
        "clinical_fact_recall_percent": 76.67,
        "critical_fact_recall_percent": 81.18,
        "negation_recall_percent": 44.0,
        "canonical_grounding_percent": 97.39,
        "compatibility_generation_seconds": 24.99,
        "final_selected": True
    },
    {
        "model": "MedGemma 1.5 4B IT",
        "structured_json_success": "1 successful compatibility configuration",
        "clinical_fact_recall_percent": None,
        "critical_fact_recall_percent": None,
        "negation_recall_percent": None,
        "canonical_grounding_percent": 90.70,
        "compatibility_generation_seconds": 133.18,
        "final_selected": False
    }
])

comparison_csv = (
    final_dir
    / "final_nlp_model_comparison.csv"
)

comparison.to_csv(
    comparison_csv,
    index=False
)


# ============================================================
# BACKUP STAGE 2
# ============================================================

backup_path = Path(
    "/home/jovyan/"
    "Case_Study_2_Final_NLP_Extraction_Backup.zip"
)

nlp_root = (
    project
    / "results/nlp"
)

files_added = 0

with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    # All NLP results
    for file_path in nlp_root.rglob("*"):

        if file_path.is_file():

            z.write(
                file_path,
                Path("results/nlp")
                / file_path.relative_to(
                    nlp_root
                )
            )

            files_added += 1

    # Gold checklist
    checklist = (
        project
        / "data/PriMock57_10_Pilot_Clinical_Gold_Checklist.csv"
    )

    if checklist.exists():

        z.write(
            checklist,
            Path("data")
            / checklist.name
        )

        files_added += 1

    # Notebooks if present
    notebook_candidates = [
        Path(
            "/home/jovyan/"
            "05_Clinical_Extraction_MedGemma.ipynb"
        ),
        Path(
            "/home/jovyan/"
            "06_Clinical_Extraction_Qwen3.ipynb"
        )
    ]

    for notebook in notebook_candidates:

        if notebook.exists():

            z.write(
                notebook,
                Path("notebooks")
                / notebook.name
            )

            files_added += 1


print("Selected extraction model: Qwen3-8B")
print("Selection status: FINAL")

print("\nFinal clinical fact recall: 76.67%")
print("Critical fact recall: 81.18%")
print("Negation recall: 44.0%")
print("Canonical evidence grounding: 97.39%")

print("\nFiles added to backup:", files_added)
print("Backup:", backup_path)

print(
    "Backup size:",
    round(
        backup_path.stat().st_size
        / 1024**2,
        2
    ),
    "MB"
)

print("\n" + "=" * 78)
print("NLP EXTRACTION STAGE: COMPLETE AND READY FOR BACKUP")
print("=" * 78)

FINAL NLP EXTRACTION STAGE FREEZE
Selected extraction model: Qwen3-8B
Selection status: FINAL

Final clinical fact recall: 76.67%
Critical fact recall: 81.18%
Negation recall: 44.0%
Canonical evidence grounding: 97.39%

Files added to backup: 72
Backup: /home/jovyan/Case_Study_2_Final_NLP_Extraction_Backup.zip
Backup size: 0.16 MB

NLP EXTRACTION STAGE: COMPLETE AND READY FOR BACKUP


In [11]:
from pathlib import Path
import json

print("=" * 75)
print("STAGE 3 — PREPARE SOAP INPUT")
print("=" * 75)

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

consultation_id = "day1_consultation01"

canonical_path = (
    project
    / "results/nlp/qwen3_8b/full_pilot/canonical"
    / f"{consultation_id}_qwen3_canonical.json"
)

soap_input_dir = (
    project
    / "results/soap/inputs"
)

soap_input_dir.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    canonical_path,
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)

facts = data.get(
    "clinical_facts",
    []
)

soap_facts = []

for i, item in enumerate(
    facts,
    start=1
):
    soap_facts.append({
        "fact_id": f"F{i:02d}",
        "category": item.get(
            "category"
        ),
        "fact": item.get(
            "fact"
        ),
        "status": item.get(
            "status"
        ),
        "importance": item.get(
            "importance"
        ),
        "evidence_quote": item.get(
            "evidence_quote"
        )
    })

soap_input = {
    "consultation_id":
        consultation_id,

    "source":
        "validated_canonical_clinical_facts",

    "clinical_facts":
        soap_facts
}

output_path = (
    soap_input_dir
    / f"{consultation_id}_soap_input.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        soap_input,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Facts prepared:",
    len(soap_facts)
)

print("\nFACT IDS")
print("-" * 75)

for item in soap_facts:
    print(
        item["fact_id"],
        "|",
        item["category"],
        "|",
        item["status"],
        "|",
        item["fact"]
    )

print("\nSaved:")
print(output_path)

print("\nSOAP INPUT PREPARATION: COMPLETE")

STAGE 3 — PREPARE SOAP INPUT
Facts prepared: 20

FACT IDS
---------------------------------------------------------------------------
F01 | presenting_complaint | present | Diarrhoea
F02 | symptom | present | Loose and watery stool
F03 | symptom | present | Frequent bowel movements
F04 | symptom | present | Lower abdominal pain
F05 | symptom | present | Cramp-like pain
F06 | symptom | present | Weakness and shakiness
F07 | symptom | present | Vomiting
F08 | symptom | present | Loss of appetite
F09 | symptom | absent | Blood in vomit
F10 | symptom | absent | Blood in stool
F11 | exposure_history | present | Chinese takeaway consumption
F12 | medical_history | present | Asthma
F13 | assessment | present | Gastroenteritis
F14 | plan | present | Hydration
F15 | plan | present | Paracetamol for fever and weakness
F16 | plan | present | Time off work
F17 | plan | present | Follow-up if symptoms persist
F18 | plan | present | Stool sample testing
F19 | social_history | absent | Smoking
F20 | 

In [12]:
from pathlib import Path
import json
import re
import time
import torch

print("=" * 78)
print("STAGE 3 — QWEN3-8B TRACEABLE SOAP GENERATION TEST")
print("=" * 78)

if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError(
        "Qwen3-8B is not loaded. Re-run the Qwen model-loading cell."
    )

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

consultation_id = "day1_consultation01"

input_path = (
    project
    / "results/soap/inputs"
    / f"{consultation_id}_soap_input.json"
)

output_dir = (
    project
    / "results/soap/qwen3_8b/compatibility_test"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    input_path,
    "r",
    encoding="utf-8"
) as f:
    source_data = json.load(f)

clinical_facts = source_data[
    "clinical_facts"
]

valid_fact_ids = {
    item["fact_id"]
    for item in clinical_facts
}

fact_categories = {
    item["fact_id"]: item["category"]
    for item in clinical_facts
}


# ============================================================
# PROMPT
# ============================================================

facts_text = json.dumps(
    clinical_facts,
    indent=2,
    ensure_ascii=False
)

prompt = f"""
Create a concise SOAP medical record draft using ONLY the
validated clinical facts below.

STRICT RULES:

1. Do not use any information that is not contained in the
   supplied clinical facts.
2. Do not invent examination findings, vital signs, tests,
   diagnoses, treatments or history.
3. Preserve negative findings correctly.
4. Every SOAP statement MUST contain one or more source_fact_ids.
5. source_fact_ids may ONLY use the supplied Fxx identifiers.
6. Do not cite a source fact that does not support the statement.
7. Keep the record concise and clinically readable.
8. This is a DRAFT requiring human approval.
9. Return valid JSON only.
10. No Markdown, reasoning or explanation.

SOAP SECTION RULES:

SUBJECTIVE:
Use patient-reported complaints, symptoms, relevant history,
medications, allergies, exposure and relevant social history.

For a source fact with status="absent", write the statement as
a denial/negative finding, for example:
"Denies blood in stool."

OBJECTIVE:
Only include explicit examination findings, measurements, test
results or clinician-observed objective information contained
in the supplied facts.

If no objective information exists, return [].
Do NOT invent normal examination findings or vital signs.

ASSESSMENT:
Use ONLY facts whose category is "assessment".

PLAN:
Use ONLY facts whose category is "plan" or "safety_netting".

Return exactly this structure:

{{
  "consultation_id": "{consultation_id}",
  "record_type": "SOAP",
  "review_status": "draft_unapproved",
  "soap": {{
    "subjective": [
      {{
        "statement": "",
        "source_fact_ids": ["F01"]
      }}
    ],
    "objective": [],
    "assessment": [
      {{
        "statement": "",
        "source_fact_ids": ["F13"]
      }}
    ],
    "plan": [
      {{
        "statement": "",
        "source_fact_ids": ["F14"]
      }}
    ]
  }}
}}

VALIDATED CLINICAL FACTS:

{facts_text}
"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

input_tokens = inputs[
    "input_ids"
].shape[-1]

print("Input tokens:", input_tokens)


# ============================================================
# GENERATE
# ============================================================

torch.cuda.reset_peak_memory_stats()

start = time.time()

with torch.inference_mode():

    output_ids = model.generate(
        **inputs,
        max_new_tokens=1800,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generation_seconds = (
    time.time() - start
)

generated_ids = output_ids[
    0,
    input_tokens:
]

generated_tokens = len(
    generated_ids
)

response = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

peak_gpu_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)


# ============================================================
# CLEAN + PARSE
# ============================================================

cleaned = response.strip()

cleaned = re.sub(
    r"^```(?:json)?\s*",
    "",
    cleaned,
    flags=re.IGNORECASE
)

cleaned = re.sub(
    r"\s*```$",
    "",
    cleaned
).strip()

first = cleaned.find("{")
last = cleaned.rfind("}")

if first >= 0 and last > first:
    json_text = cleaned[
        first:last + 1
    ]
else:
    json_text = cleaned

json_valid = False
soap = None
json_error = None

try:
    soap = json.loads(
        json_text
    )
    json_valid = True
except Exception as e:
    json_error = str(e)


# ============================================================
# TRACEABILITY VALIDATION
# ============================================================

validation_errors = []

allowed_source_categories = {
    "subjective": {
        "presenting_complaint",
        "symptom",
        "temporal_detail",
        "medication",
        "allergy",
        "medical_history",
        "exposure_history",
        "social_history"
    },
    "objective": set(),
    "assessment": {
        "assessment"
    },
    "plan": {
        "plan",
        "safety_netting"
    }
}

total_statements = 0
traceable_statements = 0

if json_valid:

    if soap.get(
        "review_status"
    ) != "draft_unapproved":
        validation_errors.append(
            "review_status must be draft_unapproved"
        )

    soap_sections = soap.get(
        "soap",
        {}
    )

    for section in [
        "subjective",
        "objective",
        "assessment",
        "plan"
    ]:

        items = soap_sections.get(
            section,
            []
        )

        if not isinstance(
            items,
            list
        ):
            validation_errors.append(
                f"{section} must be a list"
            )
            continue

        for number, item in enumerate(
            items,
            start=1
        ):

            total_statements += 1

            ids = item.get(
                "source_fact_ids",
                []
            )

            if not ids:

                validation_errors.append(
                    f"{section} item {number}: "
                    "missing source_fact_ids"
                )

                continue

            invalid_ids = [
                fact_id
                for fact_id in ids
                if fact_id
                not in valid_fact_ids
            ]

            if invalid_ids:

                validation_errors.append(
                    f"{section} item {number}: "
                    f"invalid IDs {invalid_ids}"
                )

                continue

            wrong_category_ids = []

            for fact_id in ids:

                category = fact_categories[
                    fact_id
                ]

                if category not in (
                    allowed_source_categories[
                        section
                    ]
                ):

                    wrong_category_ids.append(
                        f"{fact_id}:{category}"
                    )

            if wrong_category_ids:

                validation_errors.append(
                    f"{section} item {number}: "
                    "wrong source category "
                    f"{wrong_category_ids}"
                )

                continue

            traceable_statements += 1


# Objective should be empty for this source set
if json_valid:

    objective = soap.get(
        "soap",
        {}
    ).get(
        "objective",
        []
    )

    if len(objective) != 0:

        validation_errors.append(
            "Objective should be empty because "
            "no objective source facts were provided."
        )


# ============================================================
# SAVE
# ============================================================

raw_path = (
    output_dir
    / f"{consultation_id}_soap_raw.txt"
)

raw_path.write_text(
    response,
    encoding="utf-8"
)

soap_path = (
    output_dir
    / f"{consultation_id}_soap_draft.json"
)

if json_valid:

    with open(
        soap_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            soap,
            f,
            indent=2,
            ensure_ascii=False
        )

metadata = {
    "consultation_id":
        consultation_id,

    "model":
        "Qwen/Qwen3-8B",

    "json_valid":
        json_valid,

    "json_error":
        json_error,

    "input_tokens":
        input_tokens,

    "generated_tokens":
        generated_tokens,

    "generation_seconds":
        round(
            generation_seconds,
            2
        ),

    "peak_gpu_gb":
        round(
            peak_gpu_gb,
            2
        ),

    "total_soap_statements":
        total_statements,

    "traceable_statements":
        traceable_statements,

    "traceability_percent":
        (
            round(
                traceable_statements
                / total_statements
                * 100,
                2
            )
            if total_statements
            else None
        ),

    "validation_errors":
        validation_errors
}

metadata_path = (
    output_dir
    / f"{consultation_id}_soap_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=2
    )


# ============================================================
# OUTPUT
# ============================================================

print("\n" + "=" * 78)
print("SOAP GENERATION TEST RESULT")
print("=" * 78)

print("JSON valid:", json_valid)
print("Generated tokens:", generated_tokens)

print(
    "Hit token limit:",
    generated_tokens >= 1800
)

print(
    "SOAP statements:",
    total_statements
)

print(
    "Traceable statements:",
    traceable_statements
)

print(
    "Traceability:",
    (
        round(
            traceable_statements
            / total_statements
            * 100,
            2
        )
        if total_statements
        else 0
    ),
    "%"
)

print(
    "Validation errors:",
    len(validation_errors)
)

if validation_errors:

    print("\nVALIDATION ERRORS")

    for error in validation_errors:
        print("-", error)

if json_valid:

    print("\nSOAP DRAFT")
    print("-" * 78)

    print(
        json.dumps(
            soap,
            indent=2,
            ensure_ascii=False
        )
    )

print(
    "\nGeneration time:",
    round(
        generation_seconds,
        2
    ),
    "seconds"
)

print(
    "Peak GPU:",
    round(
        peak_gpu_gb,
        2
    ),
    "GB"
)

print("\nSaved:")
print(raw_path)

if json_valid:
    print(soap_path)

print(metadata_path)

print("\n" + "=" * 78)

if (
    json_valid
    and len(validation_errors) == 0
):

    print(
        "SOAP TRACEABILITY VALIDATION: PASS"
    )

else:

    print(
        "SOAP TRACEABILITY VALIDATION: REVIEW REQUIRED"
    )

print("=" * 78)

STAGE 3 — QWEN3-8B TRACEABLE SOAP GENERATION TEST
Input tokens: 1714

SOAP GENERATION TEST RESULT
JSON valid: True
Generated tokens: 567
Hit token limit: False
SOAP statements: 18
Traceable statements: 18
Traceability: 100.0 %
Validation errors: 0

SOAP DRAFT
------------------------------------------------------------------------------
{
  "consultation_id": "day1_consultation01",
  "record_type": "SOAP",
  "review_status": "draft_unapproved",
  "soap": {
    "subjective": [
      {
        "statement": "Diarrhoea for the last three days, affecting me.",
        "source_fact_ids": [
          "F01"
        ]
      },
      {
        "statement": "Loose and watery stool.",
        "source_fact_ids": [
          "F02"
        ]
      },
      {
        "statement": "Frequent bowel movements.",
        "source_fact_ids": [
          "F03"
        ]
      },
      {
        "statement": "Lower abdominal pain.",
        "source_fact_ids": [
          "F04"
        ]
      },
      {
      

In [13]:
from pathlib import Path
import json
import shutil
from datetime import datetime, timezone

print("=" * 78)
print("STAGE 3 — CREATE HUMAN REVIEW PACKAGE")
print("=" * 78)

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

consultation_id = "day1_consultation01"

review_dir = (
    project
    / "results/hitl/review_sessions"
    / consultation_id
)

review_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# SOURCE FILES
# ============================================================

transcript_source = (
    project
    / "results/asr/whisper_large_v3/datalab_pilot"
    / f"{consultation_id}_transcript.txt"
)

facts_source = (
    project
    / "results/nlp/qwen3_8b/full_pilot/canonical"
    / f"{consultation_id}_qwen3_canonical.json"
)

soap_source = (
    project
    / "results/soap/qwen3_8b/compatibility_test"
    / f"{consultation_id}_soap_draft.json"
)

for path in [
    transcript_source,
    facts_source,
    soap_source
]:
    if not path.exists():
        raise FileNotFoundError(path)

# ============================================================
# KEEP ORIGINAL ARTIFACTS SEPARATE
# ============================================================

transcript_target = (
    review_dir
    / "01_original_transcript.txt"
)

facts_target = (
    review_dir
    / "02_generated_clinical_facts.json"
)

soap_target = (
    review_dir
    / "03_generated_soap_draft.json"
)

shutil.copy2(
    transcript_source,
    transcript_target
)

shutil.copy2(
    facts_source,
    facts_target
)

shutil.copy2(
    soap_source,
    soap_target
)

# ============================================================
# EMPTY CORRECTION LOG
# ============================================================

correction_log = {
    "consultation_id": consultation_id,
    "log_type": "human_correction_log",
    "edits": []
}

correction_log_path = (
    review_dir
    / "04_correction_log.json"
)

with open(
    correction_log_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        correction_log,
        f,
        indent=2,
        ensure_ascii=False
    )

# ============================================================
# REVIEW STATE
# ============================================================

review_state = {
    "consultation_id": consultation_id,

    "review_status": "draft_unapproved",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "original_transcript_file":
        transcript_target.name,

    "generated_clinical_facts_file":
        facts_target.name,

    "generated_soap_draft_file":
        soap_target.name,

    "correction_log_file":
        correction_log_path.name,

    "final_approved_record_file":
        None,

    "human_actions_supported": [
        "edit_transcript",
        "edit_fact",
        "add_fact",
        "remove_fact",
        "move_soap_statement",
        "edit_soap_statement",
        "remove_soap_statement",
        "approve_record"
    ],

    "approval_required":
        True
}

review_state_path = (
    review_dir
    / "05_review_state.json"
)

with open(
    review_state_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        review_state,
        f,
        indent=2,
        ensure_ascii=False
    )

# ============================================================
# VERIFY PACKAGE
# ============================================================

print(
    "Consultation:",
    consultation_id
)

print(
    "Review status:",
    review_state[
        "review_status"
    ]
)

print(
    "Approval required:",
    review_state[
        "approval_required"
    ]
)

print(
    "Correction-log entries:",
    len(
        correction_log[
            "edits"
        ]
    )
)

print("\nREVIEW PACKAGE FILES")
print("-" * 78)

for path in sorted(
    review_dir.iterdir()
):
    print(
        path.name,
        "|",
        round(
            path.stat().st_size
            / 1024,
            2
        ),
        "KB"
    )

print("\nSaved review package:")
print(review_dir)

print("\n" + "=" * 78)
print("HUMAN REVIEW PACKAGE: READY")
print("=" * 78)

STAGE 3 — CREATE HUMAN REVIEW PACKAGE
Consultation: day1_consultation01
Review status: draft_unapproved
Approval required: True
Correction-log entries: 0

REVIEW PACKAGE FILES
------------------------------------------------------------------------------
01_original_transcript.txt | 6.81 KB
02_generated_clinical_facts.json | 4.11 KB
03_generated_soap_draft.json | 2.44 KB
04_correction_log.json | 0.1 KB
05_review_state.json | 0.65 KB

Saved review package:
/home/jovyan/Case_Study_2_Medical_Consultation_AI/results/hitl/review_sessions/day1_consultation01

HUMAN REVIEW PACKAGE: READY


In [14]:
from pathlib import Path
import json
import shutil

print("=" * 78)
print("HITL TRACEABILITY PACKAGE REPAIR")
print("=" * 78)

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

consultation_id = "day1_consultation01"

review_dir = (
    project
    / "results/hitl/review_sessions"
    / consultation_id
)

soap_input_path = (
    project
    / "results/soap/inputs"
    / f"{consultation_id}_soap_input.json"
)

soap_draft_path = (
    review_dir
    / "03_generated_soap_draft.json"
)

facts_target = (
    review_dir
    / "02_generated_clinical_facts.json"
)

# ============================================================
# LOAD FACTS WITH Fxx IDs
# ============================================================

with open(
    soap_input_path,
    "r",
    encoding="utf-8"
) as f:
    soap_input = json.load(f)

facts_with_ids = {
    "consultation_id":
        consultation_id,

    "source":
        "validated_canonical_clinical_facts",

    "clinical_facts":
        soap_input["clinical_facts"]
}

with open(
    facts_target,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        facts_with_ids,
        f,
        indent=2,
        ensure_ascii=False
    )

# ============================================================
# VERIFY SOAP -> FACT TRACEABILITY
# ============================================================

with open(
    soap_draft_path,
    "r",
    encoding="utf-8"
) as f:
    soap = json.load(f)

valid_fact_ids = {
    item["fact_id"]
    for item in facts_with_ids[
        "clinical_facts"
    ]
}

used_fact_ids = []

for section in [
    "subjective",
    "objective",
    "assessment",
    "plan"
]:
    for item in soap[
        "soap"
    ].get(section, []):

        used_fact_ids.extend(
            item.get(
                "source_fact_ids",
                []
            )
        )

missing_ids = sorted(
    set(used_fact_ids)
    - valid_fact_ids
)

print(
    "Clinical facts with IDs:",
    len(valid_fact_ids)
)

print(
    "SOAP source references:",
    len(used_fact_ids)
)

print(
    "Unique SOAP fact IDs used:",
    len(set(used_fact_ids))
)

print(
    "Missing/unresolved IDs:",
    len(missing_ids)
)

if missing_ids:
    print(
        "Missing IDs:",
        missing_ids
    )

print("\nFirst five clinical facts:")
print("-" * 78)

for item in facts_with_ids[
    "clinical_facts"
][:5]:
    print(
        item["fact_id"],
        "|",
        item["fact"]
    )

print("\nUpdated:")
print(facts_target)

print("\n" + "=" * 78)

if len(missing_ids) == 0:
    print(
        "SOAP-TO-FACT TRACEABILITY: PASS"
    )
else:
    print(
        "SOAP-TO-FACT TRACEABILITY: REVIEW REQUIRED"
    )

print("=" * 78)

HITL TRACEABILITY PACKAGE REPAIR
Clinical facts with IDs: 20
SOAP source references: 18
Unique SOAP fact IDs used: 18
Missing/unresolved IDs: 0

First five clinical facts:
------------------------------------------------------------------------------
F01 | Diarrhoea
F02 | Loose and watery stool
F03 | Frequent bowel movements
F04 | Lower abdominal pain
F05 | Cramp-like pain

Updated:
/home/jovyan/Case_Study_2_Medical_Consultation_AI/results/hitl/review_sessions/day1_consultation01/02_generated_clinical_facts.json

SOAP-TO-FACT TRACEABILITY: PASS


In [15]:
from pathlib import Path
from datetime import datetime, timezone
import json
import copy

print("=" * 78)
print("STAGE 3 — HITL EDIT AND APPROVAL WORKFLOW TEST")
print("=" * 78)

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

consultation_id = "day1_consultation01"

review_dir = (
    project
    / "results/hitl/review_sessions"
    / consultation_id
)

draft_path = (
    review_dir
    / "03_generated_soap_draft.json"
)

correction_log_path = (
    review_dir
    / "04_correction_log.json"
)

review_state_path = (
    review_dir
    / "05_review_state.json"
)

working_path = (
    review_dir
    / "06_working_review_record.json"
)

approved_path = (
    review_dir
    / "07_final_approved_record.json"
)

# ============================================================
# LOAD ORIGINAL DRAFT
# ============================================================

with open(
    draft_path,
    "r",
    encoding="utf-8"
) as f:
    original_draft = json.load(f)

working_record = copy.deepcopy(
    original_draft
)

# Keep explicit workflow-test status
working_record["review_status"] = (
    "under_human_review"
)

working_record["review_metadata"] = {
    "review_type":
        "workflow_test",

    "clinical_approval":
        False,

    "note":
        (
            "Technical HITL workflow demonstration only. "
            "This is not approval by a licensed clinician."
        )
}

# ============================================================
# LOAD CORRECTION LOG
# ============================================================

with open(
    correction_log_path,
    "r",
    encoding="utf-8"
) as f:
    correction_log = json.load(f)

# ============================================================
# SIMULATED REVIEW EDIT
# ============================================================

# Find plan item linked to F14
target_index = None

for i, item in enumerate(
    working_record["soap"]["plan"]
):
    if "F14" in item.get(
        "source_fact_ids",
        []
    ):
        target_index = i
        break

if target_index is None:
    raise RuntimeError(
        "Could not find SOAP statement linked to F14."
    )

old_statement = (
    working_record[
        "soap"
    ][
        "plan"
    ][
        target_index
    ][
        "statement"
    ]
)

new_statement = (
    "Maintain adequate hydration."
)

working_record[
    "soap"
][
    "plan"
][
    target_index
][
    "statement"
] = new_statement

edit_entry = {
    "edit_id":
        "E001",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "reviewer":
        "workflow_test_reviewer",

    "review_type":
        "simulated_workflow_test",

    "action":
        "edit_soap_statement",

    "section":
        "plan",

    "source_fact_ids":
        ["F14"],

    "before":
        old_statement,

    "after":
        new_statement,

    "reason":
        (
            "Improve readability while preserving "
            "the same supported clinical meaning."
        ),

    "clinical_approval":
        False
}

correction_log[
    "edits"
].append(
    edit_entry
)

# ============================================================
# SAVE WORKING RECORD
# ============================================================

with open(
    working_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        working_record,
        f,
        indent=2,
        ensure_ascii=False
    )

# ============================================================
# SIMULATED APPROVAL ACTION
# ============================================================

approved_record = copy.deepcopy(
    working_record
)

approved_record[
    "review_status"
] = "workflow_test_approved"

approved_record[
    "review_metadata"
][
    "approved_at_utc"
] = datetime.now(
    timezone.utc
).isoformat()

approved_record[
    "review_metadata"
][
    "approved_by"
] = "workflow_test_reviewer"

approved_record[
    "review_metadata"
][
    "clinical_approval"
] = False

approved_record[
    "review_metadata"
][
    "note"
] = (
    "Technical workflow approval only. "
    "Not reviewed or approved by a licensed clinician."
)

approval_entry = {
    "edit_id":
        "E002",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "reviewer":
        "workflow_test_reviewer",

    "review_type":
        "simulated_workflow_test",

    "action":
        "approve_record",

    "before":
        "under_human_review",

    "after":
        "workflow_test_approved",

    "clinical_approval":
        False
}

correction_log[
    "edits"
].append(
    approval_entry
)

# ============================================================
# SAVE APPROVED RECORD + LOG
# ============================================================

with open(
    approved_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        approved_record,
        f,
        indent=2,
        ensure_ascii=False
    )

with open(
    correction_log_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        correction_log,
        f,
        indent=2,
        ensure_ascii=False
    )

# ============================================================
# UPDATE REVIEW STATE
# ============================================================

with open(
    review_state_path,
    "r",
    encoding="utf-8"
) as f:
    review_state = json.load(f)

review_state[
    "review_status"
] = "workflow_test_approved"

review_state[
    "final_approved_record_file"
] = approved_path.name

review_state[
    "clinical_approval"
] = False

review_state[
    "workflow_test_completed"
] = True

with open(
    review_state_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        review_state,
        f,
        indent=2,
        ensure_ascii=False
    )

# ============================================================
# VERIFY ORIGINAL DRAFT WAS NOT ALTERED
# ============================================================

with open(
    draft_path,
    "r",
    encoding="utf-8"
) as f:
    draft_check = json.load(f)

original_still_unchanged = (
    draft_check[
        "soap"
    ][
        "plan"
    ][
        target_index
    ][
        "statement"
    ]
    == old_statement
)

# ============================================================
# OUTPUT
# ============================================================

print(
    "Original generated statement:",
    old_statement
)

print(
    "Reviewer-edited statement:",
    new_statement
)

print(
    "Original draft unchanged:",
    original_still_unchanged
)

print(
    "Correction log entries:",
    len(
        correction_log["edits"]
    )
)

print(
    "Final review status:",
    review_state[
        "review_status"
    ]
)

print(
    "Clinical approval:",
    review_state[
        "clinical_approval"
    ]
)

print("\nCORRECTION LOG")
print("-" * 78)

for edit in correction_log[
    "edits"
]:
    print(
        edit["edit_id"],
        "|",
        edit["action"],
        "|",
        edit.get(
            "before",
            ""
        ),
        "->",
        edit.get(
            "after",
            ""
        )
    )

print("\nFILES")
print("-" * 78)

for path in sorted(
    review_dir.iterdir()
):
    print(path.name)

print("\n" + "=" * 78)

if (
    original_still_unchanged
    and len(
        correction_log["edits"]
    ) == 2
    and review_state[
        "workflow_test_completed"
    ]
):
    print(
        "HITL EDIT + APPROVAL WORKFLOW: PASS"
    )
else:
    print(
        "HITL EDIT + APPROVAL WORKFLOW: REVIEW REQUIRED"
    )

print("=" * 78)

STAGE 3 — HITL EDIT AND APPROVAL WORKFLOW TEST
Original generated statement: Hydration.
Reviewer-edited statement: Maintain adequate hydration.
Original draft unchanged: True
Correction log entries: 2
Final review status: workflow_test_approved
Clinical approval: False

CORRECTION LOG
------------------------------------------------------------------------------
E001 | edit_soap_statement | Hydration. -> Maintain adequate hydration.
E002 | approve_record | under_human_review -> workflow_test_approved

FILES
------------------------------------------------------------------------------
01_original_transcript.txt
02_generated_clinical_facts.json
03_generated_soap_draft.json
04_correction_log.json
05_review_state.json
06_working_review_record.json
07_final_approved_record.json

HITL EDIT + APPROVAL WORKFLOW: PASS


In [16]:
from pathlib import Path
import json
import re
import time
import torch
import pandas as pd


print("=" * 78)
print("QWEN3-8B FULL 10-CONSULTATION SOAP GENERATION")
print("=" * 78)


# ============================================================
# VERIFY MODEL
# ============================================================

if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError(
        "Qwen3-8B is not loaded. Re-run the Qwen loading cell first."
    )


# ============================================================
# PATHS
# ============================================================

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

canonical_dir = (
    project
    / "results/nlp/qwen3_8b/full_pilot/canonical"
)

soap_input_dir = (
    project
    / "results/soap/inputs"
)

output_root = (
    project
    / "results/soap/qwen3_8b/full_pilot"
)

draft_dir = output_root / "drafts"
metadata_dir = output_root / "metadata"
raw_dir = output_root / "raw"

for folder in [
    soap_input_dir,
    draft_dir,
    metadata_dir,
    raw_dir
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


consultation_ids = [
    "day1_consultation01",
    "day1_consultation07",
    "day2_consultation01",
    "day2_consultation05",
    "day3_consultation06",
    "day3_consultation09",
    "day4_consultation03",
    "day4_consultation10",
    "day5_consultation03",
    "day5_consultation12"
]


# ============================================================
# VALIDATION RULES
# ============================================================

allowed_source_categories = {

    "subjective": {
        "presenting_complaint",
        "symptom",
        "temporal_detail",
        "medication",
        "allergy",
        "medical_history",
        "exposure_history",
        "social_history"
    },

    "objective": set(),

    "assessment": {
        "assessment"
    },

    "plan": {
        "plan",
        "safety_netting"
    }
}


# ============================================================
# RUN ALL 10
# ============================================================

summary_rows = []

batch_start = time.time()


for index, consultation_id in enumerate(
    consultation_ids,
    start=1
):

    print("\n" + "=" * 78)
    print(
        f"[{index}/10] {consultation_id}"
    )
    print("=" * 78)

    canonical_path = (
        canonical_dir
        / f"{consultation_id}_qwen3_canonical.json"
    )

    if not canonical_path.exists():

        print("ERROR: canonical facts missing")

        summary_rows.append({
            "consultation_id": consultation_id,
            "json_valid": False,
            "soap_statements": 0,
            "traceability_percent": 0,
            "validation_errors": 1,
            "generation_seconds": 0,
            "error": "canonical_facts_missing"
        })

        continue


    # ========================================================
    # LOAD CANONICAL FACTS + ASSIGN Fxx IDS
    # ========================================================

    with open(
        canonical_path,
        "r",
        encoding="utf-8"
    ) as f:
        canonical = json.load(f)

    source_facts = []

    for fact_number, item in enumerate(
        canonical.get(
            "clinical_facts",
            []
        ),
        start=1
    ):

        source_facts.append({
            "fact_id":
                f"F{fact_number:02d}",

            "category":
                item.get("category"),

            "fact":
                item.get("fact"),

            "status":
                item.get("status"),

            "importance":
                item.get("importance"),

            "evidence_quote":
                item.get("evidence_quote")
        })


    soap_input = {
        "consultation_id":
            consultation_id,

        "source":
            "validated_canonical_clinical_facts",

        "clinical_facts":
            source_facts
    }


    input_path = (
        soap_input_dir
        / f"{consultation_id}_soap_input.json"
    )

    with open(
        input_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            soap_input,
            f,
            indent=2,
            ensure_ascii=False
        )


    valid_fact_ids = {
        item["fact_id"]
        for item in source_facts
    }

    fact_categories = {
        item["fact_id"]:
            item["category"]
        for item in source_facts
    }


    # ========================================================
    # PROMPT
    # ========================================================

    facts_text = json.dumps(
        source_facts,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
Create a concise SOAP medical record draft using ONLY the
validated clinical facts below.

STRICT RULES:

1. Do not use information that is not contained in the supplied facts.
2. Do not invent examination findings, vital signs, tests,
   diagnoses, treatments or history.
3. Preserve negative findings correctly.
4. Every SOAP statement MUST contain one or more source_fact_ids.
5. source_fact_ids may ONLY use the supplied Fxx identifiers.
6. Do not cite a fact that does not support the statement.
7. Keep the record concise and clinically readable.
8. This is a draft requiring human approval.
9. Return valid JSON only.
10. No Markdown, reasoning or explanation.

SOAP RULES:

SUBJECTIVE:
Use patient-reported complaint, symptoms, history, exposure,
medications, allergies and relevant social history.

If status="absent", express it as a denial or negative finding.

OBJECTIVE:
Include only explicit objective measurements, examinations or
test results supplied in the clinical facts.

If none exist, return [].
Never invent normal findings.

ASSESSMENT:
Use only facts with category="assessment".

PLAN:
Use only facts with category="plan" or "safety_netting".

Return exactly:

{{
  "consultation_id": "{consultation_id}",
  "record_type": "SOAP",
  "review_status": "draft_unapproved",
  "soap": {{
    "subjective": [
      {{
        "statement": "",
        "source_fact_ids": ["F01"]
      }}
    ],
    "objective": [],
    "assessment": [
      {{
        "statement": "",
        "source_fact_ids": ["F01"]
      }}
    ],
    "plan": [
      {{
        "statement": "",
        "source_fact_ids": ["F01"]
      }}
    ]
  }}
}}

VALIDATED CLINICAL FACTS:

{facts_text}
"""


    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]


    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )


    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(
        model.device
    )

    input_tokens = (
        inputs["input_ids"].shape[-1]
    )


    # ========================================================
    # GENERATE
    # ========================================================

    torch.cuda.reset_peak_memory_stats()

    start = time.time()

    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=1800,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generation_seconds = (
        time.time() - start
    )

    generated_ids = output_ids[
        0,
        input_tokens:
    ]

    generated_tokens = len(
        generated_ids
    )

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    peak_gpu_gb = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )


    # ========================================================
    # SAVE RAW
    # ========================================================

    raw_path = (
        raw_dir
        / f"{consultation_id}_soap_raw.txt"
    )

    raw_path.write_text(
        response,
        encoding="utf-8"
    )


    # ========================================================
    # CLEAN + PARSE
    # ========================================================

    cleaned = response.strip()

    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned
    ).strip()

    first = cleaned.find("{")
    last = cleaned.rfind("}")

    if first >= 0 and last > first:
        json_text = cleaned[
            first:last + 1
        ]
    else:
        json_text = cleaned


    json_valid = False
    soap_record = None
    json_error = None

    try:

        soap_record = json.loads(
            json_text
        )

        json_valid = True

    except Exception as e:

        json_error = str(e)


    if not json_valid:

        print("JSON valid: False")
        print("Error:", json_error)

        summary_rows.append({
            "consultation_id": consultation_id,
            "json_valid": False,
            "soap_statements": 0,
            "traceability_percent": 0,
            "validation_errors": 1,
            "generation_seconds":
                round(generation_seconds, 2),
            "error": json_error
        })

        continue


    # ========================================================
    # TRACEABILITY VALIDATION
    # ========================================================

    validation_errors = []

    total_statements = 0
    traceable_statements = 0

    sections = soap_record.get(
        "soap",
        {}
    )


    if soap_record.get(
        "review_status"
    ) != "draft_unapproved":

        validation_errors.append(
            "review_status invalid"
        )


    for section in [
        "subjective",
        "objective",
        "assessment",
        "plan"
    ]:

        items = sections.get(
            section,
            []
        )

        if not isinstance(
            items,
            list
        ):

            validation_errors.append(
                f"{section} not a list"
            )

            continue


        for statement_number, item in enumerate(
            items,
            start=1
        ):

            total_statements += 1

            ids = item.get(
                "source_fact_ids",
                []
            )

            if not ids:

                validation_errors.append(
                    f"{section} {statement_number}: "
                    "missing source IDs"
                )

                continue


            invalid_ids = [
                fact_id
                for fact_id in ids
                if fact_id
                not in valid_fact_ids
            ]

            if invalid_ids:

                validation_errors.append(
                    f"{section} {statement_number}: "
                    f"invalid IDs {invalid_ids}"
                )

                continue


            wrong_categories = []

            for fact_id in ids:

                category = (
                    fact_categories[
                        fact_id
                    ]
                )

                if category not in (
                    allowed_source_categories[
                        section
                    ]
                ):

                    wrong_categories.append(
                        f"{fact_id}:{category}"
                    )


            if wrong_categories:

                validation_errors.append(
                    f"{section} {statement_number}: "
                    f"wrong categories {wrong_categories}"
                )

                continue


            traceable_statements += 1


    # Objective must remain empty with current fact schema
    objective = sections.get(
        "objective",
        []
    )

    if len(objective) != 0:

        validation_errors.append(
            "Objective section should be empty"
        )


    traceability = (
        round(
            traceable_statements
            / total_statements
            * 100,
            2
        )
        if total_statements
        else 0
    )


    # ========================================================
    # SAVE DRAFT
    # ========================================================

    draft_path = (
        draft_dir
        / f"{consultation_id}_soap_draft.json"
    )

    with open(
        draft_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            soap_record,
            f,
            indent=2,
            ensure_ascii=False
        )


    # ========================================================
    # METADATA
    # ========================================================

    metadata = {

        "consultation_id":
            consultation_id,

        "model":
            "Qwen/Qwen3-8B",

        "json_valid":
            True,

        "source_facts":
            len(source_facts),

        "soap_statements":
            total_statements,

        "traceable_statements":
            traceable_statements,

        "traceability_percent":
            traceability,

        "validation_errors":
            validation_errors,

        "input_tokens":
            input_tokens,

        "generated_tokens":
            generated_tokens,

        "hit_token_limit":
            generated_tokens >= 1800,

        "generation_seconds":
            round(
                generation_seconds,
                2
            ),

        "peak_gpu_gb":
            round(
                peak_gpu_gb,
                2
            )
    }


    metadata_path = (
        metadata_dir
        / f"{consultation_id}_soap_metadata.json"
    )

    with open(
        metadata_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            metadata,
            f,
            indent=2,
            ensure_ascii=False
        )


    print(
        "JSON valid: True"
    )

    print(
        "SOAP statements:",
        total_statements
    )

    print(
        "Traceability:",
        traceability,
        "%"
    )

    print(
        "Validation errors:",
        len(validation_errors)
    )

    print(
        "Generation:",
        round(
            generation_seconds,
            2
        ),
        "sec"
    )


    summary_rows.append({

        "consultation_id":
            consultation_id,

        "json_valid":
            True,

        "source_facts":
            len(source_facts),

        "soap_statements":
            total_statements,

        "traceable_statements":
            traceable_statements,

        "traceability_percent":
            traceability,

        "validation_errors":
            len(validation_errors),

        "hit_token_limit":
            generated_tokens >= 1800,

        "generation_seconds":
            round(
                generation_seconds,
                2
            ),

        "peak_gpu_gb":
            round(
                peak_gpu_gb,
                2
            ),

        "error":
            None
    })


# ============================================================
# SUMMARY
# ============================================================

batch_seconds = (
    time.time()
    - batch_start
)

summary_df = pd.DataFrame(
    summary_rows
)

summary_csv = (
    output_root
    / "qwen3_8b_full_soap_summary.csv"
)

summary_df.to_csv(
    summary_csv,
    index=False
)


summary_json = (
    output_root
    / "qwen3_8b_full_soap_summary.json"
)

with open(
    summary_json,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "model":
                "Qwen/Qwen3-8B",

            "consultations":
                10,

            "batch_wall_seconds":
                round(
                    batch_seconds,
                    2
                ),

            "results":
                summary_df.to_dict(
                    orient="records"
                )
        },
        f,
        indent=2
    )


print("\n" + "=" * 78)
print("FULL SOAP GENERATION SUMMARY")
print("=" * 78)

display_columns = [
    "consultation_id",
    "json_valid",
    "source_facts",
    "soap_statements",
    "traceability_percent",
    "validation_errors",
    "generation_seconds"
]

print(
    summary_df[
        display_columns
    ].to_string(
        index=False
    )
)


print("\nOVERALL")
print("-" * 78)

print(
    "Valid JSON:",
    f'{int(summary_df["json_valid"].sum())}/10'
)

print(
    "Average traceability:",
    round(
        summary_df[
            "traceability_percent"
        ].mean(),
        2
    ),
    "%"
)

print(
    "Total validation errors:",
    int(
        summary_df[
            "validation_errors"
        ].sum()
    )
)

print(
    "Total SOAP statements:",
    int(
        summary_df[
            "soap_statements"
        ].sum()
    )
)

print(
    "Total generation time:",
    round(
        summary_df[
            "generation_seconds"
        ].sum(),
        2
    ),
    "seconds"
)

print(
    "Batch wall time:",
    round(
        batch_seconds,
        2
    ),
    "seconds"
)

print("\nSaved:")
print(summary_csv)
print(summary_json)

print("\n" + "=" * 78)
print("FULL SOAP PILOT: COMPLETE")
print("=" * 78)

QWEN3-8B FULL 10-CONSULTATION SOAP GENERATION

[1/10] day1_consultation01
JSON valid: True
SOAP statements: 3
Traceability: 100.0 %
Validation errors: 0
Generation: 7.93 sec

[2/10] day1_consultation07
JSON valid: True
SOAP statements: 3
Traceability: 100.0 %
Validation errors: 0
Generation: 7.29 sec

[3/10] day2_consultation01
JSON valid: True
SOAP statements: 20
Traceability: 100.0 %
Validation errors: 0
Generation: 14.21 sec

[4/10] day2_consultation05
JSON valid: True
SOAP statements: 20
Traceability: 100.0 %
Validation errors: 0
Generation: 14.56 sec

[5/10] day3_consultation06
JSON valid: True
SOAP statements: 3
Traceability: 100.0 %
Validation errors: 0
Generation: 5.14 sec

[6/10] day3_consultation09
JSON valid: True
SOAP statements: 16
Traceability: 93.75 %
Validation errors: 1
Generation: 14.41 sec

[7/10] day4_consultation03
JSON valid: True
SOAP statements: 3
Traceability: 100.0 %
Validation errors: 0
Generation: 6.22 sec

[8/10] day4_consultation10
JSON valid: True
SOAP st

In [17]:
from pathlib import Path
import json
import re
import time
import torch
import pandas as pd


print("=" * 78)
print("QWEN3-8B FULL 10-CONSULTATION SOAP GENERATION")
print("=" * 78)


# ============================================================
# VERIFY MODEL
# ============================================================

if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError(
        "Qwen3-8B is not loaded. Re-run the Qwen loading cell first."
    )


# ============================================================
# PATHS
# ============================================================

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

canonical_dir = (
    project
    / "results/nlp/qwen3_8b/full_pilot/canonical"
)

soap_input_dir = (
    project
    / "results/soap/inputs"
)

output_root = (
    project
    / "results/soap/qwen3_8b/full_pilot"
)

draft_dir = output_root / "drafts"
metadata_dir = output_root / "metadata"
raw_dir = output_root / "raw"

for folder in [
    soap_input_dir,
    draft_dir,
    metadata_dir,
    raw_dir
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


consultation_ids = [
    "day1_consultation01",
    "day1_consultation07",
    "day2_consultation01",
    "day2_consultation05",
    "day3_consultation06",
    "day3_consultation09",
    "day4_consultation03",
    "day4_consultation10",
    "day5_consultation03",
    "day5_consultation12"
]


# ============================================================
# VALIDATION RULES
# ============================================================

allowed_source_categories = {

    "subjective": {
        "presenting_complaint",
        "symptom",
        "temporal_detail",
        "medication",
        "allergy",
        "medical_history",
        "exposure_history",
        "social_history"
    },

    "objective": set(),

    "assessment": {
        "assessment"
    },

    "plan": {
        "plan",
        "safety_netting"
    }
}


# ============================================================
# RUN ALL 10
# ============================================================

summary_rows = []

batch_start = time.time()


for index, consultation_id in enumerate(
    consultation_ids,
    start=1
):

    print("\n" + "=" * 78)
    print(
        f"[{index}/10] {consultation_id}"
    )
    print("=" * 78)

    canonical_path = (
        canonical_dir
        / f"{consultation_id}_qwen3_canonical.json"
    )

    if not canonical_path.exists():

        print("ERROR: canonical facts missing")

        summary_rows.append({
            "consultation_id": consultation_id,
            "json_valid": False,
            "soap_statements": 0,
            "traceability_percent": 0,
            "validation_errors": 1,
            "generation_seconds": 0,
            "error": "canonical_facts_missing"
        })

        continue


    # ========================================================
    # LOAD CANONICAL FACTS + ASSIGN Fxx IDS
    # ========================================================

    with open(
        canonical_path,
        "r",
        encoding="utf-8"
    ) as f:
        canonical = json.load(f)

    source_facts = []

    for fact_number, item in enumerate(
        canonical.get(
            "clinical_facts",
            []
        ),
        start=1
    ):

        source_facts.append({
            "fact_id":
                f"F{fact_number:02d}",

            "category":
                item.get("category"),

            "fact":
                item.get("fact"),

            "status":
                item.get("status"),

            "importance":
                item.get("importance"),

            "evidence_quote":
                item.get("evidence_quote")
        })


    soap_input = {
        "consultation_id":
            consultation_id,

        "source":
            "validated_canonical_clinical_facts",

        "clinical_facts":
            source_facts
    }


    input_path = (
        soap_input_dir
        / f"{consultation_id}_soap_input.json"
    )

    with open(
        input_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            soap_input,
            f,
            indent=2,
            ensure_ascii=False
        )


    valid_fact_ids = {
        item["fact_id"]
        for item in source_facts
    }

    fact_categories = {
        item["fact_id"]:
            item["category"]
        for item in source_facts
    }


    # ========================================================
    # PROMPT
    # ========================================================

    facts_text = json.dumps(
        source_facts,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
Create a concise SOAP medical record draft using ONLY the
validated clinical facts below.

STRICT RULES:

1. Do not use information that is not contained in the supplied facts.
2. Do not invent examination findings, vital signs, tests,
   diagnoses, treatments or history.
3. Preserve negative findings correctly.
4. Every SOAP statement MUST contain one or more source_fact_ids.
5. source_fact_ids may ONLY use the supplied Fxx identifiers.
6. Do not cite a fact that does not support the statement.
7. Keep the record concise and clinically readable.
8. This is a draft requiring human approval.
9. Return valid JSON only.
10. No Markdown, reasoning or explanation.

SOAP RULES:

SUBJECTIVE:
Use patient-reported complaint, symptoms, history, exposure,
medications, allergies and relevant social history.

If status="absent", express it as a denial or negative finding.

OBJECTIVE:
Include only explicit objective measurements, examinations or
test results supplied in the clinical facts.

If none exist, return [].
Never invent normal findings.

ASSESSMENT:
Use only facts with category="assessment".

PLAN:
Use only facts with category="plan" or "safety_netting".

Return exactly:

{{
  "consultation_id": "{consultation_id}",
  "record_type": "SOAP",
  "review_status": "draft_unapproved",
  "soap": {{
    "subjective": [
      {{
        "statement": "",
        "source_fact_ids": ["F01"]
      }}
    ],
    "objective": [],
    "assessment": [
      {{
        "statement": "",
        "source_fact_ids": ["F01"]
      }}
    ],
    "plan": [
      {{
        "statement": "",
        "source_fact_ids": ["F01"]
      }}
    ]
  }}
}}

VALIDATED CLINICAL FACTS:

{facts_text}
"""


    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]


    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )


    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(
        model.device
    )

    input_tokens = (
        inputs["input_ids"].shape[-1]
    )


    # ========================================================
    # GENERATE
    # ========================================================

    torch.cuda.reset_peak_memory_stats()

    start = time.time()

    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=1800,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generation_seconds = (
        time.time() - start
    )

    generated_ids = output_ids[
        0,
        input_tokens:
    ]

    generated_tokens = len(
        generated_ids
    )

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    peak_gpu_gb = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )


    # ========================================================
    # SAVE RAW
    # ========================================================

    raw_path = (
        raw_dir
        / f"{consultation_id}_soap_raw.txt"
    )

    raw_path.write_text(
        response,
        encoding="utf-8"
    )


    # ========================================================
    # CLEAN + PARSE
    # ========================================================

    cleaned = response.strip()

    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned
    ).strip()

    first = cleaned.find("{")
    last = cleaned.rfind("}")

    if first >= 0 and last > first:
        json_text = cleaned[
            first:last + 1
        ]
    else:
        json_text = cleaned


    json_valid = False
    soap_record = None
    json_error = None

    try:

        soap_record = json.loads(
            json_text
        )

        json_valid = True

    except Exception as e:

        json_error = str(e)


    if not json_valid:

        print("JSON valid: False")
        print("Error:", json_error)

        summary_rows.append({
            "consultation_id": consultation_id,
            "json_valid": False,
            "soap_statements": 0,
            "traceability_percent": 0,
            "validation_errors": 1,
            "generation_seconds":
                round(generation_seconds, 2),
            "error": json_error
        })

        continue


    # ========================================================
    # TRACEABILITY VALIDATION
    # ========================================================

    validation_errors = []

    total_statements = 0
    traceable_statements = 0

    sections = soap_record.get(
        "soap",
        {}
    )


    if soap_record.get(
        "review_status"
    ) != "draft_unapproved":

        validation_errors.append(
            "review_status invalid"
        )


    for section in [
        "subjective",
        "objective",
        "assessment",
        "plan"
    ]:

        items = sections.get(
            section,
            []
        )

        if not isinstance(
            items,
            list
        ):

            validation_errors.append(
                f"{section} not a list"
            )

            continue


        for statement_number, item in enumerate(
            items,
            start=1
        ):

            total_statements += 1

            ids = item.get(
                "source_fact_ids",
                []
            )

            if not ids:

                validation_errors.append(
                    f"{section} {statement_number}: "
                    "missing source IDs"
                )

                continue


            invalid_ids = [
                fact_id
                for fact_id in ids
                if fact_id
                not in valid_fact_ids
            ]

            if invalid_ids:

                validation_errors.append(
                    f"{section} {statement_number}: "
                    f"invalid IDs {invalid_ids}"
                )

                continue


            wrong_categories = []

            for fact_id in ids:

                category = (
                    fact_categories[
                        fact_id
                    ]
                )

                if category not in (
                    allowed_source_categories[
                        section
                    ]
                ):

                    wrong_categories.append(
                        f"{fact_id}:{category}"
                    )


            if wrong_categories:

                validation_errors.append(
                    f"{section} {statement_number}: "
                    f"wrong categories {wrong_categories}"
                )

                continue


            traceable_statements += 1


    # Objective must remain empty with current fact schema
    objective = sections.get(
        "objective",
        []
    )

    if len(objective) != 0:

        validation_errors.append(
            "Objective section should be empty"
        )


    traceability = (
        round(
            traceable_statements
            / total_statements
            * 100,
            2
        )
        if total_statements
        else 0
    )


    # ========================================================
    # SAVE DRAFT
    # ========================================================

    draft_path = (
        draft_dir
        / f"{consultation_id}_soap_draft.json"
    )

    with open(
        draft_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            soap_record,
            f,
            indent=2,
            ensure_ascii=False
        )


    # ========================================================
    # METADATA
    # ========================================================

    metadata = {

        "consultation_id":
            consultation_id,

        "model":
            "Qwen/Qwen3-8B",

        "json_valid":
            True,

        "source_facts":
            len(source_facts),

        "soap_statements":
            total_statements,

        "traceable_statements":
            traceable_statements,

        "traceability_percent":
            traceability,

        "validation_errors":
            validation_errors,

        "input_tokens":
            input_tokens,

        "generated_tokens":
            generated_tokens,

        "hit_token_limit":
            generated_tokens >= 1800,

        "generation_seconds":
            round(
                generation_seconds,
                2
            ),

        "peak_gpu_gb":
            round(
                peak_gpu_gb,
                2
            )
    }


    metadata_path = (
        metadata_dir
        / f"{consultation_id}_soap_metadata.json"
    )

    with open(
        metadata_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            metadata,
            f,
            indent=2,
            ensure_ascii=False
        )


    print(
        "JSON valid: True"
    )

    print(
        "SOAP statements:",
        total_statements
    )

    print(
        "Traceability:",
        traceability,
        "%"
    )

    print(
        "Validation errors:",
        len(validation_errors)
    )

    print(
        "Generation:",
        round(
            generation_seconds,
            2
        ),
        "sec"
    )


    summary_rows.append({

        "consultation_id":
            consultation_id,

        "json_valid":
            True,

        "source_facts":
            len(source_facts),

        "soap_statements":
            total_statements,

        "traceable_statements":
            traceable_statements,

        "traceability_percent":
            traceability,

        "validation_errors":
            len(validation_errors),

        "hit_token_limit":
            generated_tokens >= 1800,

        "generation_seconds":
            round(
                generation_seconds,
                2
            ),

        "peak_gpu_gb":
            round(
                peak_gpu_gb,
                2
            ),

        "error":
            None
    })


# ============================================================
# SUMMARY
# ============================================================

batch_seconds = (
    time.time()
    - batch_start
)

summary_df = pd.DataFrame(
    summary_rows
)

summary_csv = (
    output_root
    / "qwen3_8b_full_soap_summary.csv"
)

summary_df.to_csv(
    summary_csv,
    index=False
)


summary_json = (
    output_root
    / "qwen3_8b_full_soap_summary.json"
)

with open(
    summary_json,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "model":
                "Qwen/Qwen3-8B",

            "consultations":
                10,

            "batch_wall_seconds":
                round(
                    batch_seconds,
                    2
                ),

            "results":
                summary_df.to_dict(
                    orient="records"
                )
        },
        f,
        indent=2
    )


print("\n" + "=" * 78)
print("FULL SOAP GENERATION SUMMARY")
print("=" * 78)

display_columns = [
    "consultation_id",
    "json_valid",
    "source_facts",
    "soap_statements",
    "traceability_percent",
    "validation_errors",
    "generation_seconds"
]

print(
    summary_df[
        display_columns
    ].to_string(
        index=False
    )
)


print("\nOVERALL")
print("-" * 78)

print(
    "Valid JSON:",
    f'{int(summary_df["json_valid"].sum())}/10'
)

print(
    "Average traceability:",
    round(
        summary_df[
            "traceability_percent"
        ].mean(),
        2
    ),
    "%"
)

print(
    "Total validation errors:",
    int(
        summary_df[
            "validation_errors"
        ].sum()
    )
)

print(
    "Total SOAP statements:",
    int(
        summary_df[
            "soap_statements"
        ].sum()
    )
)

print(
    "Total generation time:",
    round(
        summary_df[
            "generation_seconds"
        ].sum(),
        2
    ),
    "seconds"
)

print(
    "Batch wall time:",
    round(
        batch_seconds,
        2
    ),
    "seconds"
)

print("\nSaved:")
print(summary_csv)
print(summary_json)

print("\n" + "=" * 78)
print("FULL SOAP PILOT: COMPLETE")
print("=" * 78)

QWEN3-8B FULL 10-CONSULTATION SOAP GENERATION

[1/10] day1_consultation01
JSON valid: True
SOAP statements: 3
Traceability: 100.0 %
Validation errors: 0
Generation: 7.63 sec

[2/10] day1_consultation07
JSON valid: True
SOAP statements: 3
Traceability: 100.0 %
Validation errors: 0
Generation: 7.0 sec

[3/10] day2_consultation01
JSON valid: True
SOAP statements: 20
Traceability: 100.0 %
Validation errors: 0
Generation: 13.31 sec

[4/10] day2_consultation05
JSON valid: True
SOAP statements: 20
Traceability: 100.0 %
Validation errors: 0
Generation: 14.38 sec

[5/10] day3_consultation06
JSON valid: True
SOAP statements: 3
Traceability: 100.0 %
Validation errors: 0
Generation: 5.01 sec

[6/10] day3_consultation09
JSON valid: True
SOAP statements: 16
Traceability: 93.75 %
Validation errors: 1
Generation: 14.04 sec

[7/10] day4_consultation03
JSON valid: True
SOAP statements: 3
Traceability: 100.0 %
Validation errors: 0
Generation: 5.96 sec

[8/10] day4_consultation10
JSON valid: True
SOAP sta

In [18]:
from pathlib import Path
import json

print("=" * 78)
print("SOAP PILOT — SUSPICIOUS CASE INSPECTION")
print("=" * 78)

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

output_root = (
    project
    / "results/soap/qwen3_8b/full_pilot"
)

draft_dir = output_root / "drafts"
metadata_dir = output_root / "metadata"

suspicious_cases = [
    "day1_consultation01",
    "day1_consultation07",
    "day3_consultation06",
    "day3_consultation09",
    "day4_consultation03",
    "day4_consultation10",
    "day5_consultation03",
    "day5_consultation12"
]

for consultation_id in suspicious_cases:

    print("\n" + "=" * 78)
    print(consultation_id)
    print("=" * 78)

    metadata_path = (
        metadata_dir
        / f"{consultation_id}_soap_metadata.json"
    )

    draft_path = (
        draft_dir
        / f"{consultation_id}_soap_draft.json"
    )

    if metadata_path.exists():

        with open(
            metadata_path,
            "r",
            encoding="utf-8"
        ) as f:
            metadata = json.load(f)

        print(
            "Source facts:",
            metadata.get("source_facts")
        )

        print(
            "SOAP statements:",
            metadata.get("soap_statements")
        )

        print(
            "Traceability:",
            metadata.get("traceability_percent"),
            "%"
        )

        print(
            "Validation errors:",
            metadata.get("validation_errors")
        )

    else:
        print("Metadata missing")

    if draft_path.exists():

        with open(
            draft_path,
            "r",
            encoding="utf-8"
        ) as f:
            draft = json.load(f)

        soap = draft.get(
            "soap",
            {}
        )

        print("\nSECTION COUNTS")

        for section in [
            "subjective",
            "objective",
            "assessment",
            "plan"
        ]:

            items = soap.get(
                section,
                []
            )

            print(
                section,
                ":",
                len(items)
            )

        print("\nFULL SOAP DRAFT")
        print("-" * 78)

        print(
            json.dumps(
                draft,
                indent=2,
                ensure_ascii=False
            )
        )

    else:
        print("SOAP draft missing")

print("\n" + "=" * 78)
print("SUSPICIOUS SOAP INSPECTION: COMPLETE")
print("=" * 78)

SOAP PILOT — SUSPICIOUS CASE INSPECTION

day1_consultation01
Source facts: 20
SOAP statements: 3
Traceability: 100.0 %
Validation errors: []

SECTION COUNTS
subjective : 1
objective : 0
assessment : 1
plan : 1

FULL SOAP DRAFT
------------------------------------------------------------------------------
{
  "consultation_id": "day1_consultation01",
  "record_type": "SOAP",
  "review_status": "draft_unapproved",
  "soap": {
    "subjective": [
      {
        "statement": "Patient reports diarrhoea for the last three days, affecting them. Symptoms include loose and watery stool, frequent bowel movements, lower abdominal pain, cramp-like pain, weakness and shakiness, vomiting, and loss of appetite. No blood in vomit or stool. Consumed Chinese takeaway. History of asthma. No smoking or alcohol consumption.",
        "source_fact_ids": [
          "F01",
          "F02",
          "F03",
          "F04",
          "F05",
          "F06",
          "F07",
          "F08",
          "F09",


In [19]:
from pathlib import Path
import json
import pandas as pd

print("=" * 78)
print("FULL SOAP SOURCE-FACT COVERAGE AUDIT")
print("=" * 78)

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

input_dir = (
    project
    / "results/soap/inputs"
)

draft_dir = (
    project
    / "results/soap/qwen3_8b/full_pilot/drafts"
)

output_dir = (
    project
    / "results/soap/qwen3_8b/evaluation"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

consultation_ids = [
    "day1_consultation01",
    "day1_consultation07",
    "day2_consultation01",
    "day2_consultation05",
    "day3_consultation06",
    "day3_consultation09",
    "day4_consultation03",
    "day4_consultation10",
    "day5_consultation03",
    "day5_consultation12"
]

allowed_categories = {
    "subjective": {
        "presenting_complaint",
        "symptom",
        "temporal_detail",
        "medication",
        "allergy",
        "medical_history",
        "exposure_history",
        "social_history"
    },
    "objective": set(),
    "assessment": {
        "assessment"
    },
    "plan": {
        "plan",
        "safety_netting"
    }
}

summary_rows = []
detail_rows = []

for consultation_id in consultation_ids:

    input_path = (
        input_dir
        / f"{consultation_id}_soap_input.json"
    )

    draft_path = (
        draft_dir
        / f"{consultation_id}_soap_draft.json"
    )

    with open(
        input_path,
        "r",
        encoding="utf-8"
    ) as f:
        soap_input = json.load(f)

    with open(
        draft_path,
        "r",
        encoding="utf-8"
    ) as f:
        draft = json.load(f)

    facts = soap_input[
        "clinical_facts"
    ]

    fact_lookup = {
        item["fact_id"]: item
        for item in facts
    }

    all_fact_ids = set(
        fact_lookup.keys()
    )

    used_ids = set()
    wrong_section_refs = []

    for section in [
        "subjective",
        "objective",
        "assessment",
        "plan"
    ]:

        for statement_number, item in enumerate(
            draft["soap"].get(
                section,
                []
            ),
            start=1
        ):

            for fact_id in item.get(
                "source_fact_ids",
                []
            ):

                used_ids.add(
                    fact_id
                )

                if fact_id not in fact_lookup:

                    wrong_section_refs.append({
                        "section":
                            section,

                        "statement_number":
                            statement_number,

                        "fact_id":
                            fact_id,

                        "category":
                            "UNKNOWN",

                        "problem":
                            "invalid_fact_id"
                    })

                    continue

                category = fact_lookup[
                    fact_id
                ].get(
                    "category"
                )

                if category not in (
                    allowed_categories[
                        section
                    ]
                ):

                    wrong_section_refs.append({
                        "section":
                            section,

                        "statement_number":
                            statement_number,

                        "fact_id":
                            fact_id,

                        "category":
                            category,

                        "problem":
                            "wrong_section_category"
                    })

    missing_ids = sorted(
        all_fact_ids
        - used_ids
    )

    valid_used_ids = (
        used_ids
        & all_fact_ids
    )

    coverage_percent = round(
        len(
            valid_used_ids
        )
        / len(
            all_fact_ids
        )
        * 100,
        2
    )

    print("\n" + consultation_id)
    print("-" * 78)

    print(
        "Source facts:",
        len(all_fact_ids)
    )

    print(
        "Referenced source facts:",
        len(valid_used_ids)
    )

    print(
        "Source-fact coverage:",
        coverage_percent,
        "%"
    )

    print(
        "Missing facts:",
        len(missing_ids)
    )

    if missing_ids:

        for fact_id in missing_ids:

            item = fact_lookup[
                fact_id
            ]

            print(
                "  ",
                fact_id,
                "|",
                item.get("category"),
                "|",
                item.get("status"),
                "|",
                item.get("fact")
            )

            detail_rows.append({
                "consultation_id":
                    consultation_id,

                "fact_id":
                    fact_id,

                "category":
                    item.get(
                        "category"
                    ),

                "status":
                    item.get(
                        "status"
                    ),

                "fact":
                    item.get(
                        "fact"
                    ),

                "issue":
                    "fact_not_referenced"
            })

    print(
        "Wrong-section references:",
        len(wrong_section_refs)
    )

    for problem in wrong_section_refs:

        print(
            "  ",
            problem["section"],
            "|",
            problem["fact_id"],
            "|",
            problem["category"]
        )

        detail_rows.append({
            "consultation_id":
                consultation_id,

            "fact_id":
                problem["fact_id"],

            "category":
                problem["category"],

            "status":
                None,

            "fact":
                fact_lookup.get(
                    problem["fact_id"],
                    {}
                ).get(
                    "fact"
                ),

            "issue":
                (
                    "wrong_section:"
                    + problem[
                        "section"
                    ]
                )
        })

    summary_rows.append({
        "consultation_id":
            consultation_id,

        "source_facts":
            len(
                all_fact_ids
            ),

        "referenced_facts":
            len(
                valid_used_ids
            ),

        "fact_coverage_percent":
            coverage_percent,

        "missing_facts":
            len(
                missing_ids
            ),

        "wrong_section_references":
            len(
                wrong_section_refs
            )
    })


summary_df = pd.DataFrame(
    summary_rows
)

detail_df = pd.DataFrame(
    detail_rows
)

summary_path = (
    output_dir
    / "soap_source_fact_coverage_summary.csv"
)

detail_path = (
    output_dir
    / "soap_source_fact_coverage_issues.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)

detail_df.to_csv(
    detail_path,
    index=False
)

print("\n" + "=" * 78)
print("SOAP SOURCE-FACT COVERAGE SUMMARY")
print("=" * 78)

print(
    summary_df.to_string(
        index=False
    )
)

print("\nOVERALL")
print("-" * 78)

print(
    "Average source-fact coverage:",
    round(
        summary_df[
            "fact_coverage_percent"
        ].mean(),
        2
    ),
    "%"
)

print(
    "Total missing facts:",
    int(
        summary_df[
            "missing_facts"
        ].sum()
    )
)

print(
    "Total wrong-section references:",
    int(
        summary_df[
            "wrong_section_references"
        ].sum()
    )
)

print("\nSaved:")
print(summary_path)
print(detail_path)

print("\n" + "=" * 78)
print("SOAP SOURCE-FACT COVERAGE AUDIT: COMPLETE")
print("=" * 78)

FULL SOAP SOURCE-FACT COVERAGE AUDIT

day1_consultation01
------------------------------------------------------------------------------
Source facts: 20
Referenced source facts: 20
Source-fact coverage: 100.0 %
Missing facts: 0
Wrong-section references: 0

day1_consultation07
------------------------------------------------------------------------------
Source facts: 20
Referenced source facts: 20
Source-fact coverage: 100.0 %
Missing facts: 0
Wrong-section references: 0

day2_consultation01
------------------------------------------------------------------------------
Source facts: 20
Referenced source facts: 20
Source-fact coverage: 100.0 %
Missing facts: 0
Wrong-section references: 0

day2_consultation05
------------------------------------------------------------------------------
Source facts: 20
Referenced source facts: 20
Source-fact coverage: 100.0 %
Missing facts: 0
Wrong-section references: 0

day3_consultation06
--------------------------------------------------------------

In [20]:
from pathlib import Path
import json
import copy
import pandas as pd

print("=" * 78)
print("STAGE 3 — DETERMINISTIC SOAP VALIDATOR / POSTPROCESSOR")
print("=" * 78)

project = Path(
    "/home/jovyan/Case_Study_2_Medical_Consultation_AI"
)

input_dir = (
    project
    / "results/soap/inputs"
)

raw_draft_dir = (
    project
    / "results/soap/qwen3_8b/full_pilot/drafts"
)

canonical_dir = (
    project
    / "results/soap/qwen3_8b/canonical"
)

repair_log_dir = (
    project
    / "results/soap/qwen3_8b/repair_logs"
)

evaluation_dir = (
    project
    / "results/soap/qwen3_8b/evaluation"
)

for folder in [
    canonical_dir,
    repair_log_dir,
    evaluation_dir
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


consultation_ids = [
    "day1_consultation01",
    "day1_consultation07",
    "day2_consultation01",
    "day2_consultation05",
    "day3_consultation06",
    "day3_consultation09",
    "day4_consultation03",
    "day4_consultation10",
    "day5_consultation03",
    "day5_consultation12"
]


allowed_categories = {

    "subjective": {
        "presenting_complaint",
        "symptom",
        "temporal_detail",
        "medication",
        "allergy",
        "medical_history",
        "exposure_history",
        "social_history"
    },

    "objective": set(),

    "assessment": {
        "assessment"
    },

    "plan": {
        "plan",
        "safety_netting"
    }
}


category_to_section = {}

for section, categories in (
    allowed_categories.items()
):

    for category in categories:
        category_to_section[
            category
        ] = section


# ============================================================
# SAFE DETERMINISTIC RENDERING
# ============================================================

def render_missing_fact(item):

    fact = str(
        item.get(
            "fact",
            ""
        )
    ).strip()

    category = str(
        item.get(
            "category",
            ""
        )
    ).strip()

    status = str(
        item.get(
            "status",
            "present"
        )
    ).strip().lower()

    evidence = str(
        item.get(
            "evidence_quote",
            ""
        )
    ).strip()

    if not fact:
        fact = "Clinical fact"

    # Preserve explicit negation
    if status == "absent":

        lowered = fact.lower()

        if lowered.startswith(
            (
                "no ",
                "denies ",
                "without ",
                "absence "
            )
        ):
            statement = fact

        else:
            statement = (
                "No "
                + fact[0].lower()
                + fact[1:]
            )

    # Temporal facts benefit from their grounded quote
    elif (
        category == "temporal_detail"
        and evidence
    ):

        statement = (
            f"{fact}: {evidence}"
        )

    else:

        statement = fact

    statement = statement.strip()

    if statement and statement[-1] not in ".!?":
        statement += "."

    return statement


# ============================================================
# PROCESS ALL 10
# ============================================================

summary_rows = []


for consultation_id in consultation_ids:

    print("\n" + "=" * 78)
    print(consultation_id)
    print("=" * 78)

    input_path = (
        input_dir
        / f"{consultation_id}_soap_input.json"
    )

    draft_path = (
        raw_draft_dir
        / f"{consultation_id}_soap_draft.json"
    )

    with open(
        input_path,
        "r",
        encoding="utf-8"
    ) as f:
        soap_input = json.load(f)

    with open(
        draft_path,
        "r",
        encoding="utf-8"
    ) as f:
        raw_draft = json.load(f)


    source_facts = (
        soap_input[
            "clinical_facts"
        ]
    )

    fact_lookup = {
        item["fact_id"]: item
        for item in source_facts
    }

    all_fact_ids = set(
        fact_lookup.keys()
    )


    canonical = copy.deepcopy(
        raw_draft
    )

    canonical[
        "review_status"
    ] = "draft_unapproved"

    canonical[
        "generation_stage"
    ] = (
        "qwen_draft_plus_"
        "deterministic_validation"
    )


    repairs = []

    removed_statements = 0


    # ========================================================
    # REMOVE STRUCTURALLY INVALID STATEMENTS
    # ========================================================

    for section in [
        "subjective",
        "objective",
        "assessment",
        "plan"
    ]:

        original_items = (
            canonical[
                "soap"
            ].get(
                section,
                []
            )
        )

        cleaned_items = []


        for statement_number, item in enumerate(
            original_items,
            start=1
        ):

            ids = item.get(
                "source_fact_ids",
                []
            )

            problems = []


            if not ids:

                problems.append(
                    "missing_source_fact_ids"
                )


            for fact_id in ids:

                if fact_id not in fact_lookup:

                    problems.append(
                        f"invalid_id:{fact_id}"
                    )

                    continue


                category = (
                    fact_lookup[
                        fact_id
                    ].get(
                        "category"
                    )
                )

                if category not in (
                    allowed_categories[
                        section
                    ]
                ):

                    problems.append(
                        f"wrong_category:"
                        f"{fact_id}:"
                        f"{category}"
                    )


            if problems:

                repairs.append({
                    "action":
                        "remove_invalid_soap_statement",

                    "section":
                        section,

                    "original_statement_number":
                        statement_number,

                    "statement":
                        item.get(
                            "statement"
                        ),

                    "source_fact_ids":
                        ids,

                    "reason":
                        problems
                })

                removed_statements += 1

                continue


            cleaned_items.append(
                item
            )


        canonical[
            "soap"
        ][
            section
        ] = cleaned_items


    # ========================================================
    # FIND FACTS STILL REPRESENTED
    # ========================================================

    used_fact_ids = set()


    for section in [
        "subjective",
        "objective",
        "assessment",
        "plan"
    ]:

        for item in (
            canonical[
                "soap"
            ].get(
                section,
                []
            )
        ):

            for fact_id in item.get(
                "source_fact_ids",
                []
            ):

                if fact_id in fact_lookup:

                    used_fact_ids.add(
                        fact_id
                    )


    missing_ids = sorted(
        all_fact_ids
        - used_fact_ids
    )


    # ========================================================
    # ADD OMITTED FACTS DETERMINISTICALLY
    # ========================================================

    added_facts = 0


    for fact_id in missing_ids:

        source_fact = (
            fact_lookup[
                fact_id
            ]
        )

        category = (
            source_fact.get(
                "category"
            )
        )

        target_section = (
            category_to_section.get(
                category
            )
        )


        if target_section is None:

            repairs.append({
                "action":
                    "unable_to_place_fact",

                "fact_id":
                    fact_id,

                "category":
                    category,

                "reason":
                    "No deterministic SOAP section mapping"
            })

            continue


        statement = (
            render_missing_fact(
                source_fact
            )
        )


        canonical[
            "soap"
        ][
            target_section
        ].append({
            "statement":
                statement,

            "source_fact_ids":
                [fact_id],

            "added_by":
                "deterministic_postprocessor"
        })


        repairs.append({
            "action":
                "restore_omitted_source_fact",

            "fact_id":
                fact_id,

            "category":
                category,

            "target_section":
                target_section,

            "statement":
                statement
        })


        added_facts += 1


    # ========================================================
    # FINAL VALIDATION
    # ========================================================

    final_used_ids = set()

    final_wrong_sections = []

    final_invalid_ids = []


    for section in [
        "subjective",
        "objective",
        "assessment",
        "plan"
    ]:

        for item in (
            canonical[
                "soap"
            ].get(
                section,
                []
            )
        ):

            for fact_id in item.get(
                "source_fact_ids",
                []
            ):

                if fact_id not in fact_lookup:

                    final_invalid_ids.append(
                        fact_id
                    )

                    continue


                final_used_ids.add(
                    fact_id
                )


                category = (
                    fact_lookup[
                        fact_id
                    ].get(
                        "category"
                    )
                )


                if category not in (
                    allowed_categories[
                        section
                    ]
                ):

                    final_wrong_sections.append(
                        {
                            "section":
                                section,

                            "fact_id":
                                fact_id,

                            "category":
                                category
                        }
                    )


    final_missing_ids = sorted(
        all_fact_ids
        - final_used_ids
    )


    coverage = round(
        len(
            final_used_ids
        )
        / len(
            all_fact_ids
        )
        * 100,
        2
    )


    # ========================================================
    # SAVE
    # ========================================================

    canonical_path = (
        canonical_dir
        / f"{consultation_id}_canonical_soap.json"
    )


    with open(
        canonical_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            canonical,
            f,
            indent=2,
            ensure_ascii=False
        )


    repair_log = {
        "consultation_id":
            consultation_id,

        "original_qwen_draft":
            draft_path.name,

        "removed_invalid_statements":
            removed_statements,

        "restored_omitted_facts":
            added_facts,

        "repairs":
            repairs,

        "final_source_fact_coverage_percent":
            coverage,

        "final_missing_fact_ids":
            final_missing_ids,

        "final_wrong_section_references":
            final_wrong_sections,

        "final_invalid_fact_ids":
            final_invalid_ids
    }


    repair_path = (
        repair_log_dir
        / f"{consultation_id}_soap_repair_log.json"
    )


    with open(
        repair_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            repair_log,
            f,
            indent=2,
            ensure_ascii=False
        )


    print(
        "Removed invalid statements:",
        removed_statements
    )

    print(
        "Restored omitted facts:",
        added_facts
    )

    print(
        "Final source-fact coverage:",
        coverage,
        "%"
    )

    print(
        "Final missing facts:",
        len(
            final_missing_ids
        )
    )

    print(
        "Final wrong-section refs:",
        len(
            final_wrong_sections
        )
    )

    print(
        "Final invalid IDs:",
        len(
            final_invalid_ids
        )
    )


    summary_rows.append({

        "consultation_id":
            consultation_id,

        "source_facts":
            len(
                all_fact_ids
            ),

        "removed_invalid_statements":
            removed_statements,

        "restored_omitted_facts":
            added_facts,

        "final_fact_coverage_percent":
            coverage,

        "final_missing_facts":
            len(
                final_missing_ids
            ),

        "final_wrong_section_references":
            len(
                final_wrong_sections
            ),

        "final_invalid_fact_ids":
            len(
                final_invalid_ids
            )
    })


# ============================================================
# FINAL SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)


summary_csv = (
    evaluation_dir
    / "canonical_soap_validation_summary.csv"
)

summary_df.to_csv(
    summary_csv,
    index=False
)


print("\n" + "=" * 78)
print("CANONICAL SOAP VALIDATION SUMMARY")
print("=" * 78)


print(
    summary_df.to_string(
        index=False
    )
)


print("\nOVERALL")
print("-" * 78)


print(
    "Average final fact coverage:",
    round(
        summary_df[
            "final_fact_coverage_percent"
        ].mean(),
        2
    ),
    "%"
)


print(
    "Total restored facts:",
    int(
        summary_df[
            "restored_omitted_facts"
        ].sum()
    )
)


print(
    "Invalid SOAP statements removed:",
    int(
        summary_df[
            "removed_invalid_statements"
        ].sum()
    )
)


print(
    "Remaining missing facts:",
    int(
        summary_df[
            "final_missing_facts"
        ].sum()
    )
)


print(
    "Remaining wrong-section references:",
    int(
        summary_df[
            "final_wrong_section_references"
        ].sum()
    )
)


print(
    "Remaining invalid IDs:",
    int(
        summary_df[
            "final_invalid_fact_ids"
        ].sum()
    )
)


print("\nSaved:")
print(summary_csv)


print("\n" + "=" * 78)

if (
    summary_df[
        "final_missing_facts"
    ].sum() == 0
    and
    summary_df[
        "final_wrong_section_references"
    ].sum() == 0
    and
    summary_df[
        "final_invalid_fact_ids"
    ].sum() == 0
):

    print(
        "CANONICAL SOAP STRUCTURAL VALIDATION: PASS"
    )

else:

    print(
        "CANONICAL SOAP STRUCTURAL VALIDATION: REVIEW REQUIRED"
    )

print("=" * 78)

STAGE 3 — DETERMINISTIC SOAP VALIDATOR / POSTPROCESSOR

day1_consultation01
Removed invalid statements: 0
Restored omitted facts: 0
Final source-fact coverage: 100.0 %
Final missing facts: 0
Final wrong-section refs: 0
Final invalid IDs: 0

day1_consultation07
Removed invalid statements: 0
Restored omitted facts: 0
Final source-fact coverage: 100.0 %
Final missing facts: 0
Final wrong-section refs: 0
Final invalid IDs: 0

day2_consultation01
Removed invalid statements: 0
Restored omitted facts: 0
Final source-fact coverage: 100.0 %
Final missing facts: 0
Final wrong-section refs: 0
Final invalid IDs: 0

day2_consultation05
Removed invalid statements: 0
Restored omitted facts: 0
Final source-fact coverage: 100.0 %
Final missing facts: 0
Final wrong-section refs: 0
Final invalid IDs: 0

day3_consultation06
Removed invalid statements: 0
Restored omitted facts: 8
Final source-fact coverage: 100.0 %
Final missing facts: 0
Final wrong-section refs: 0
Final invalid IDs: 0

day3_consultation09